# Compression Analysis: Longitudinal Modulus and Cooperative Diffusivity

## Geometry

**z** = compression direction (piston normal; gel spans $z_\text{gel,lo}$ to $z_\text{gel,hi}$) | Support fixed at base | Piston moves at constant velocity until $\varepsilon_{zz} = \Delta L / L_0$ reaches `comp_percent`, then freezes for stress relaxation.

## LAMMPS Stress Outputs — What Is What

The simulation writes three classes of per-bin ($z$-direction) stress profiles.

**Partial stresses** (from `compute stress/atom`):

| Variable | LAMMPS compute | Physical content |
|---|---|---|
| $\sigma_{p,\alpha\beta}(z)$ | `stress/atom` on *polymer* group | polymer-polymer (WCA+FENE) **+** polymer-solvent virial |
| $\sigma_{s,\alpha\beta}(z)$ | `stress/atom` on *solvent* group | solvent-solvent **+** 1/2 polymer-solvent virial |

Files: `sigmazz_polymer_*.dat`, `sigmazz_solvent_*.dat`, etc.

**Solvent-solvent-only stress** (post-processed from `compute pair/local`):

| Variable | Source | Physical content |
|---|---|---|
| $\sigma_{s,ss,\alpha\beta}(z)$ | `pair/local fx fy fz` -> `pairs_*.dump` | solvent-solvent virial **only** |

`compute stress/atom` in LAMMPS 22 Jul 2025 does **not** support sub-style pair filtering (`pair <style>` is rejected at runtime).  Instead, `compute pair/local fx fy fz` records the raw force on atom $I$ for every pair in which $I$ is a solvent atom.  Post-processing reconstructs the per-pair virial
$$W_{\alpha\beta}^\text{pair} = -r_\alpha\,F_\beta$$
and bins ss pairs ($\text{type}_I = \text{type}_J = 3$) by midpoint $z_\text{mid} = (z_I+z_J)/2$:
$$\sigma_{s,ss,\alpha\beta}(z_\text{bin}) = \frac{1}{L_x L_y \Delta z}\sum_{\substack{(I,J)\text{ ss}\\ z_\text{mid}\in\text{bin}}} W_{\alpha\beta}^\text{pair}$$

## Theory

### Poroelastic Decomposition (Terzaghi)

$$\boxed{\sigma_{zz}^t(z) = \sigma'_{zz}(z) + p_p(z)}$$

LAMMPS sign convention: positive = compressive.

**Pore pressure** (preferred — uses ss-only virial; no polymer-solvent contamination):
$$p_p(z) = \frac{1}{3}\,\frac{\mathrm{tr}\,\boldsymbol{\sigma}_{s,ss}(z)}{\phi_s(z)}$$

*Fallback* when `pairs_*.dump` is absent: replace $\boldsymbol{\sigma}_{s,ss}$ with the group-based $\boldsymbol{\sigma}_s$.  The extra polymer-solvent term is approximately strain-independent, so the reference subtraction $\Delta\sigma' = \sigma'_\text{final} - \sigma'_\text{ref}$ largely cancels it.

**Effective (network) stress**:
$$\sigma'_{zz}(z) = \sigma_{p,zz}(z) + \sigma_{s,zz}(z) - p_p(z)$$

Note: $\sigma_s$ (not $\sigma_{s,ss}$) enters $\sigma'$ because the group-based solvent virial is the correct momentum-flux term in the mixture-theory stress balance; only the pore pressure requires the ss-only decomposition to avoid double-counting the polymer-solvent coupling.

### Longitudinal Modulus — Two Methods

**Method 1 — Poroelastic stress decomposition (Voronoi):**
$$M_\text{Voronoi} = \frac{\Delta\sigma'_{zz}}{\varepsilon_{zz}}, \qquad \Delta\sigma'_{zz}(z) = \sigma'_{zz,\text{final}}(z) - \sigma'_{zz,\text{ref}}(z)$$

Volume fractions $\phi_p$, $\phi_s$ are computed via Voronoi tessellation on the final relaxed frame. The CI on $M_\text{Voronoi}$ reflects spatial heterogeneity across gel bins.

**Method 2 — Piston force:**
$$M_\text{piston} = \frac{P_\text{piston}}{\varepsilon_{zz}}, \qquad P_\text{piston} = \frac{F_{z,\text{piston}}}{L_x L_y}$$

$F_{z,\text{piston}}$ is the total pairwise z-force on piston atoms (`compute reduce sum fz`, recorded via `fix print`). For a homogeneous gel at mechanical equilibrium the two estimates should agree; deviations signal spatial heterogeneity or incomplete relaxation.

### Cooperative Diffusivity — Displacement Formulation

Polymer volume fraction $\phi_p$ is proportional to polymer z-displacement $u_z$, so the cooperative diffusion PDE
$$\frac{\partial \phi_p}{\partial t} = D_c \frac{\partial^2 \phi_p}{\partial z^2}$$
is equivalent to
$$\frac{\partial u_z}{\partial t} = D_c \frac{\partial^2 u_z}{\partial z^2}$$

The displacement-based approach avoids per-frame Voronoi tessellation. $u_z(z,t)$ is recorded via `compute displace/atom` + `fix ave/chunk`, with reference positions set at Phase 2 start so $u_z(z,0) = 0$ by construction.

$D_c$ is extracted by fitting the even-mode Fourier cosine solution (wall symmetry eliminates odd modes):
$$u_z(z, t) = u_z^0 + (u_z^1 - u_z^0)\left[1 + 2\sum_{k=1}^{N} \cos\!\left(\frac{2\pi k\, z}{L_\text{gel}}\right) e^{-4\pi^2 k^2 \tau_\text{eff}}\right], \qquad \tau_\text{eff} = \frac{D_c\, t}{L_\text{gel}^2} + \frac{1}{16\pi}$$

where $u_z^0 = 0$ (support boundary; set by reference construction) and $u_z^1$ is the gel-mean displacement at the final snapshot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
from scipy import stats
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
import os
from pathlib import Path
import matplotlib.font_manager as fm

plt.rcParams.update({
    'font.family': 'CMU Serif',
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'CMU Serif',
    'font.size': 20,
    'axes.titlesize': 22,
    'axes.labelsize': 25,
    'xtick.labelsize': 23,
    'ytick.labelsize': 23,
    'legend.fontsize': 23,
    'figure.titlesize': 22,
    'axes.unicode_minus': False
})

print('Imports successful')

## Configuration

In [ ]:
# ── CONFIG: only change these lines to switch datasets ─────────────────────
RUN_ID   = "rho04_p1.52_600k_4M_1"   # folder name inside flow_data_local/compression/
sim_name = "walled_slab_support_5beads_tall_rho04_p1.52_1.0_1.0_600000_1.0_1.0_4000000"
# ───────────────────────────────────────────────────────────────────────────

from pathlib import Path
DATA_DIR  = Path("../../flow_data_local/compression") / RUN_ID
PLOT_DIR  = Path("../../flow_data_local/plots/compression") / RUN_ID
TRAJ_FILE = Path("../../flow_data_local/traj_files.nosync") / f"gel_flow_{sim_name}_1.lammpstrj"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ── Analysis parameters ────────────────────────────────────────────────────
binWidth          = 2.0
kT                = 1.0  # LJ temperature (used for kinetic stress correction
                          #   on σ_s,ss: pair/local dumps don't carry the
                          #   m·v_a·v_b kinetic term, so we add n_s(z)*kT
                          #   per diagonal component when loading)
phi_gel_threshold = 0.1
Ncount_min        = 200    # min atoms/bin for displacement gel-mask
ci_level          = 0.95
dt_lj             = 0.005  # LJ timestep
comp_percent      = 0.1
piston_area       = None   # sigma^2; auto-read from box_dimensions file;
                           # set manually here if that file is unavailable
# -- Pair/local dumps (for sigma_s,ss computation) ------------------------------
# Sync from the cluster alongside the other output files:
#   output_files/pair_data/pairs_*.dump               → DATA_DIR/
#   output_files/pair_data/pairs_ref_*.dump           → DATA_DIR/
#   traj_files/traj_stress_*.lammpstrj                → traj_files.nosync/ (same dir as TRAJ_FILE)
#   traj_files/traj_ref_*.lammpstrj                   → traj_files.nosync/
# Update these paths if you store the dumps in a different location.
PAIRS_FILE       = DATA_DIR / f'pairs_{sim_name}.dump'
PAIRS_REF_FILE   = DATA_DIR / f'pairs_ref_{sim_name}.dump'
TRAJ_STRESS_FILE = Path("../../flow_data_local/traj_files.nosync") / f"traj_stress_{sim_name}.lammpstrj"
TRAJ_REF_FILE    = Path("../../flow_data_local/traj_files.nosync") / f"traj_ref_{sim_name}.lammpstrj"

# -- Chemical potential (cavity-Widom) output files ----
MU_SUMMARY_FILE = DATA_DIR / f'mu_z_cavity_summary_{sim_name}.dat'
MU_FRAME_FILE   = DATA_DIR / f'mu_z_cavity_{sim_name}.dat'
P_EXT           = 1.52   # barostat target pressure (ε/σ³); matches p_ext in cavity_widom.py
                         # (update to match your --p-ext flag if you used a different value)

## Helper Functions

In [ ]:
def read_print_file(filepath, col_names=None):
    """Read a LAMMPS fix print output (one row per timestep).
    Lines beginning with # are skipped.
    Returns a dict of column arrays keyed by col_names (or col_0, col_1, ...).
    """
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            rows.append([float(v) for v in line.split()])
    if not rows:
        raise ValueError(f'No data in {filepath}')
    arr = np.array(rows)
    if col_names is None:
        col_names = [f'col_{i}' for i in range(arr.shape[1])]
    return {name: arr[:, i] for i, name in enumerate(col_names)}


def read_ave_time_file(filepath):
    """Read a LAMMPS fix ave/time mode vector output.
    Returns list of (timestep, bin_indices, values_array).
    """
    data_by_time = []
    with open(filepath, 'r') as f:
        lines = [l for l in f if not l.startswith('#') and l.strip()]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) == 2:
            timestep, nrows = int(parts[0]), int(parts[1])
            values = []
            for j in range(1, nrows + 1):
                if i + j < len(lines):
                    vp = lines[i + j].split()
                    if len(vp) == 2:
                        values.append(float(vp[1]))
            if values:
                data_by_time.append((timestep, np.arange(1, len(values)+1), np.array(values)))
            i += nrows + 1
        else:
            i += 1
    return data_by_time


def read_ave_chunk_file(filepath):
    """Read a LAMMPS fix ave/chunk output.
    Returns list of (timestep, chunk_array) where chunk_array columns are:
    [0]=chunk_id  [1]=Coord1  [2]=Ncount  [3]=val1  ...
    Header line format: timestep nchunks [total-count]  (2 or 3 values).
    """
    snapshots = []
    with open(filepath) as f:
        lines = [l for l in f if l.strip() and not l.startswith('#')]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) in (2, 3):
            try:
                timestep, nchunks = int(parts[0]), int(parts[1])
            except ValueError:
                i += 1
                continue
            rows = []
            for j in range(1, nchunks + 1):
                if i + j < len(lines):
                    rows.append([float(v) for v in lines[i + j].split()])
            if rows:
                snapshots.append((timestep, np.array(rows)))
            i += nchunks + 1
        else:
            i += 1
    return snapshots


def read_strain_file(filepath):
    rows = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            rows.append([float(v) for v in line.split()])
    if not rows:
        raise ValueError(f'No data in {filepath}')
    if len(rows) == 2 and all(len(r)==1 for r in rows):
        return None, np.array([rows[0][0]]), np.array([rows[1][0]])
    if len(rows) == 1 and len(rows[0]) == 2:
        return None, np.array([rows[0][0]]), np.array([rows[0][1]])
    if all(len(r)==3 for r in rows):
        arr = np.array(rows)
        ts, col1, col2 = arr[:,0].astype(int).tolist(), arr[:,1], arr[:,2]
        ratio = np.nanmedian(col2) / np.nanmedian(col1)
        dL = col2 if ratio < 0.5 else col1 - col2
        return ts, col1, dL
    raise ValueError(f'Unrecognised format in {filepath}')


def read_lammpstrj_frame(filepath, frame_idx=0):
    with open(filepath) as f:
        lines = f.readlines()
    starts = [i for i,l in enumerate(lines) if 'ITEM: TIMESTEP' in l]
    if frame_idx < 0: frame_idx = len(starts) + frame_idx
    s = starts[frame_idx]
    e = starts[frame_idx+1] if frame_idx+1<len(starts) else len(lines)
    fl = lines[s:e]
    timestep = int(fl[1].strip())
    box = {'x': [float(v) for v in fl[5].split()],
           'y': [float(v) for v in fl[6].split()],
           'z': [float(v) for v in fl[7].split()]}
    atoms = [[int(p[0]),int(p[1]),int(p[2]),float(p[3]),float(p[4]),float(p[5])]
             for p in [l.split() for l in fl[9:9+int(fl[3].strip())]]]
    return timestep, box, atoms


def compute_volume_fractions_1d(atoms_data, box_bounds, bin_width, direction='z'):
    a = np.array(atoms_data)
    typ, pos = a[:,1].astype(int), a[:,3:6]
    pm, sm = (typ==1)|(typ==2), (typ==3)
    d = {'x':0,'y':1,'z':2}[direction]
    lo, hi = box_bounds[direction]
    edges = np.arange(lo, hi+bin_width, bin_width)
    centers = (edges[:-1]+edges[1:])/2
    phi_p, phi_s = np.zeros(len(centers)), np.zeros(len(centers))
    for i,(a_,b_) in enumerate(zip(edges[:-1],edges[1:])):
        mask = (pos[:,d]>=a_)&(pos[:,d]<b_)
        np_, ns_, nt = np.sum(pm&mask), np.sum(sm&mask), np.sum(mask)
        if nt>0: phi_p[i], phi_s[i] = np_/nt, ns_/nt
    return centers, phi_p, phi_s


def compute_volume_fractions_1d_voronoi(atoms_data, box_bounds, bin_width, direction='z'):
    import tess
    import time

    a = np.array(atoms_data)
    typ = a[:,1].astype(int)
    pos = a[:,3:6]

    xlo, xhi = box_bounds['x']
    ylo, yhi = box_bounds['y']
    zlo, zhi = box_bounds['z']
    L = np.array([xhi-xlo, yhi-ylo, zhi-zlo])
    origin = np.array([xlo, ylo, zlo])

    mobile_mask = (typ == 1) | (typ == 2) | (typ == 3)
    typ_m = typ[mobile_mask]
    pos_m = pos[mobile_mask]
    pos_wrapped = origin + (pos_m - origin) % L

    print(f'  Checking for duplicate positions...')
    _, unique_idx = np.unique(np.round(pos_wrapped, 6), axis=0, return_index=True)
    n_dupes = len(pos_wrapped) - len(unique_idx)
    if n_dupes > 0:
        print(f'  Warning: removing {n_dupes} duplicate positions')
        pos_wrapped = pos_wrapped[unique_idx]
        typ_m = typ_m[unique_idx]

    print(f'  Building Voronoi container ({len(pos_wrapped)} atoms)...')
    t0 = time.time()
    cntr = tess.Container(
        pos_wrapped,
        limits=((xlo, ylo, zlo), (xhi, yhi, zhi)),
        periodic=True
    )
    print(f'  Container built ({time.time()-t0:.1f}s)')

    print(f'  Extracting cell volumes...')
    t0 = time.time()
    voronoi_volumes = np.array([c.volume() for c in cntr])
    print(f'  Volumes extracted ({time.time()-t0:.1f}s)')

    box_vol = L[0] * L[1] * L[2]
    voro_total = np.sum(voronoi_volumes)
    if abs(voro_total - box_vol) / box_vol > 0.01:
        print(f'  Warning: Voronoi volume {voro_total:.2f} differs from box {box_vol:.2f} by '
              f'{100*abs(voro_total-box_vol)/box_vol:.1f}%')

    d = {'x': 0, 'y': 1, 'z': 2}[direction]
    lo, hi = box_bounds[direction]
    edges   = np.arange(lo, hi + bin_width, bin_width)
    centers = (edges[:-1] + edges[1:]) / 2

    pm = (typ_m == 1) | (typ_m == 2)
    sm = (typ_m == 3)

    phi_p = np.zeros(len(centers))
    phi_s = np.zeros(len(centers))

    for i, (a_, b_) in enumerate(zip(edges[:-1], edges[1:])):
        mask    = (pos_wrapped[:, d] >= a_) & (pos_wrapped[:, d] < b_)
        V_p     = np.sum(voronoi_volumes[mask & pm])
        V_s     = np.sum(voronoi_volumes[mask & sm])
        V_total = np.sum(voronoi_volumes[mask])
        if V_total > 0:
            phi_p[i] = V_p / V_total
            phi_s[i] = V_s / V_total

    return centers, phi_p, phi_s


def mean_ci(values, ci_level=0.95):
    v = values[~np.isnan(values)]
    n = len(v)
    if n==0: return np.nan, np.nan, np.nan
    if n==1: return v[0], v[0], v[0]
    m = np.mean(v)
    lo, hi = stats.t.interval(ci_level, df=n-1, loc=m, scale=stats.sem(v))
    return m, lo, hi


print('Helper functions defined')

# == pair/local helpers =======================================================

def read_pairs_local_dump(filepath):
    # Read a LAMMPS dump local file (pair/local output from slab_with_flow.lmp).
    #
    # Column order set by the dump command in slab_with_flow.lmp:
    #   c_ids_local[1] = patom1    c_ids_local[2] = patom2
    #   c_ids_local[3] = ptype1    c_ids_local[4] = ptype2
    #   c_pl_local[1]  = fx        c_pl_local[2]  = fy
    #   c_pl_local[3]  = fz
    # where fx/fy/fz is the force on atom I due to atom J.
    #
    # Returns list of (timestep, box_dict, data_array) where data_array has
    # shape (n_entries, 7): [id1, id2, type1, type2, fx, fy, fz].
    frames = []
    with open(filepath) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if lines[i].strip() == 'ITEM: TIMESTEP':
            timestep  = int(lines[i+1].strip())
            n_entries = int(lines[i+3].strip())
            xlo, xhi  = map(float, lines[i+5].split())
            ylo, yhi  = map(float, lines[i+6].split())
            zlo, zhi  = map(float, lines[i+7].split())
            box = {'x': (xlo, xhi), 'y': (ylo, yhi), 'z': (zlo, zhi)}
            data_start = i + 9       # line i+8 is ITEM: ENTRIES header
            rows = []
            for j in range(n_entries):
                if data_start + j < len(lines):
                    rows.append([float(v) for v in lines[data_start + j].split()])
            data = np.array(rows) if rows else np.zeros((0, 7))
            frames.append((timestep, box, data))
            i = data_start + n_entries
        else:
            i += 1
    return frames


def compute_ss_stress_from_pairs(pairs_file, traj_file, binWidth, filter_type=3):
    # Compute solvent-solvent-only stress profiles from pair/local forces.
    #
    # For each ss pair (both atom types == filter_type):
    #   r_z = z_J - z_I  (PBC-corrected, I->J)
    #   W_zz = -(r_z * fz)   <- positive = compressive, matches LAMMPS stress/atom
    #   W_xx = -(r_x * fx),  W_yy = -(r_y * fy)
    #   Assign to z-bin by midpoint z_mid = (z_I + z_J) / 2
    #   sigma_ss_zz(bin) = sum(W_zz) / (lx * ly * binWidth)
    #
    # Also returns n_s_z (solvent number density per z-bin, per frame) so the
    # caller can apply the kinetic stress correction σ_kin,s = n_s(z)*kT that
    # is *not* present in the pair/local virial.
    #
    # Parameters
    # ----------
    # pairs_file  : path to pairs_*.dump  (dump local from slab_with_flow.lmp)
    # traj_file   : path to gel_flow_*.lammpstrj  (main trajectory)
    # binWidth    : bin width in sigma
    # filter_type : atom type for solvent (default 3)
    #
    # Returns dict with keys:
    #   timesteps, z_bins, sigma_ss_zz/xx/yy (n_frames, n_bins),
    #   n_s_z (n_frames, n_bins) — solvent number density per bin per frame,
    #   lx, ly
    print('  Reading pairs dump ...')
    pairs_frames = read_pairs_local_dump(pairs_file)
    print(f'  {len(pairs_frames)} frames in pairs dump')
    if not pairs_frames:
        return {'timesteps': [], 'z_bins': None,
                'sigma_ss_zz': np.array([]), 'sigma_ss_xx': np.array([]),
                'sigma_ss_yy': np.array([]), 'n_s_z': np.array([]),
                'lx': None, 'ly': None}

    target_ts = {ts for ts, _, _ in pairs_frames}

    # stream traj file; only keep solvent positions for timesteps we need
    print('  Streaming trajectory (solvent atoms only) ...')
    traj_by_ts = {}
    with open(traj_file) as f:
        lines = f.readlines()
    i = 0
    while i < len(lines):
        if 'ITEM: TIMESTEP' in lines[i]:
            ts = int(lines[i+1].strip())
            n_atoms = int(lines[i+3].strip())
            if ts in target_ts:
                xlo, xhi = map(float, lines[i+5].split())
                ylo, yhi = map(float, lines[i+6].split())
                zlo, zhi = map(float, lines[i+7].split())
                box = {'x': (xlo, xhi), 'y': (ylo, yhi), 'z': (zlo, zhi)}
                pos = {}
                for j in range(n_atoms):
                    p = lines[i+9+j].split()
                    if int(p[1]) == filter_type:
                        pos[int(p[0])] = (float(p[3]), float(p[4]), float(p[5]))
                traj_by_ts[ts] = (box, pos)
            i += 9 + n_atoms
        else:
            i += 1
    print(f'  {len(traj_by_ts)} matching traj frames found')

    ts_out, sxx_all, syy_all, szz_all = [], [], [], []
    ns_all = []                                 # solvent n_s(z) per frame
    z_bins_out, last_lx, last_ly = None, None, None

    for ts, box, pair_data in pairs_frames:
        if ts not in traj_by_ts:
            print(f'  WARNING: ts={ts} missing from traj -- skipped')
            continue
        _, pos = traj_by_ts[ts]

        lx = box['x'][1] - box['x'][0]
        ly = box['y'][1] - box['y'][0]
        lz = box['z'][1] - box['z'][0]
        zlo_b = box['z'][0]
        n_bins  = int(round(lz / binWidth))
        bin_vol = lx * ly * binWidth
        z_bins  = zlo_b + (np.arange(n_bins) + 0.5) * binWidth

        p_xx = np.zeros(n_bins)
        p_yy = np.zeros(n_bins)
        p_zz = np.zeros(n_bins)

        if len(pair_data) > 0:
            ss = ((pair_data[:, 2].astype(int) == filter_type) &
                  (pair_data[:, 3].astype(int) == filter_type))
            sp = pair_data[ss]
            if len(sp) > 0:
                ids1 = sp[:, 0].astype(int)
                ids2 = sp[:, 1].astype(int)
                fx, fy, fz = sp[:, 4], sp[:, 5], sp[:, 6]

                # vectorised position lookup
                all_ids = set(ids1.tolist() + ids2.tolist()) & set(pos.keys())
                id_to_idx = {aid: k for k, aid in enumerate(all_ids)}
                pos_arr = np.array([pos[aid] for aid in all_ids])

                valid = np.array([(i1 in id_to_idx and i2 in id_to_idx)
                                  for i1, i2 in zip(ids1, ids2)])
                if valid.any():
                    ids1v = ids1[valid]; ids2v = ids2[valid]
                    fxv = fx[valid]; fyv = fy[valid]; fzv = fz[valid]
                    i1 = np.array([id_to_idx[a] for a in ids1v])
                    i2 = np.array([id_to_idx[a] for a in ids2v])
                    p1 = pos_arr[i1]; p2 = pos_arr[i2]

                    dr = p2 - p1
                    dr[:, 0] -= lx * np.round(dr[:, 0] / lx)
                    dr[:, 1] -= ly * np.round(dr[:, 1] / ly)
                    dr[:, 2] -= lz * np.round(dr[:, 2] / lz)

                    w_xx = -(dr[:, 0] * fxv)
                    w_yy = -(dr[:, 1] * fyv)
                    w_zz = -(dr[:, 2] * fzv)

                    z_mids = p1[:, 2] + dr[:, 2] * 0.5
                    bidx   = ((z_mids - zlo_b) / binWidth).astype(int)
                    ok     = (bidx >= 0) & (bidx < n_bins)
                    np.add.at(p_xx, bidx[ok], w_xx[ok])
                    np.add.at(p_yy, bidx[ok], w_yy[ok])
                    np.add.at(p_zz, bidx[ok], w_zz[ok])

        # Solvent number-density profile for this frame (used for kinetic
        # correction n_s(z)*kT in the caller).  `pos` is the solvent-only
        # position dict already built for the virial assignment above.
        n_s_z = np.zeros(n_bins)
        if pos:
            solv_z = np.fromiter((p[2] for p in pos.values()),
                                 dtype=float, count=len(pos))
            bidx_n = ((solv_z - zlo_b) / binWidth).astype(int)
            ok_n   = (bidx_n >= 0) & (bidx_n < n_bins)
            np.add.at(n_s_z, bidx_n[ok_n], 1.0)
        n_s_z /= bin_vol

        ts_out.append(ts)
        sxx_all.append(p_xx / bin_vol)
        syy_all.append(p_yy / bin_vol)
        szz_all.append(p_zz / bin_vol)
        ns_all.append(n_s_z)
        z_bins_out = z_bins
        last_lx, last_ly = lx, ly

    print(f'  Done -- {len(ts_out)} frames processed')
    return {
        'timesteps':   ts_out,
        'z_bins':      z_bins_out,
        'sigma_ss_xx': np.array(sxx_all),
        'sigma_ss_yy': np.array(syy_all),
        'sigma_ss_zz': np.array(szz_all),
        'n_s_z':       np.array(ns_all),
        'lx': last_lx, 'ly': last_ly,
    }


print('Pair/local helper functions defined')

def read_mu_summary(filepath):
    """Read cavity-Widom time-averaged summary file.

    Columns (7): z_center  z_lo  z_hi  mu_ex_mean  mu_ex_stderr  p_cav_mean  n_frames
    Column  (8): rho_s_mean  — optional, present in cavity_widom.py >= 2026-05-27.
                 Solvent number density on the Widom z-grid (same bins, piston
                 excluded).  Use this instead of the packing-fraction ρ_s to
                 avoid the piston-face density spike in the stress trajectory.

    Returns dict with numpy arrays for each column.
    """
    data = []
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            vals = line.split()
            data.append([float(v) if v != 'nan' else np.nan for v in vals])
    if not data:
        raise ValueError(f'No data in {filepath}')
    arr = np.array(data)
    result = {
        'z_center':    arr[:, 0],
        'z_lo':        arr[:, 1],
        'z_hi':        arr[:, 2],
        'mu_ex_mean':  arr[:, 3],
        'mu_ex_stderr':arr[:, 4],
        'p_cav_mean':  arr[:, 5],
        'n_frames':    arr[:, 6].astype(int),
    }
    # Column 8: rho_s_mean (optional — backward-compatible with older files)
    if arr.shape[1] >= 8:
        result['rho_s_mean'] = arr[:, 7]
    return result


print('Chemical potential helper defined')

## Step 1: Load All Data

In [ ]:
# ── Stress profiles ────────────────────────────────────────────────────────
stress_data = {}
for key, fp in [
    ('polymer_z', DATA_DIR / f'stress_z_polymer_{sim_name}.dat'),
    ('solvent_z', DATA_DIR / f'stress_z_solvent_{sim_name}.dat'),
    ('polymer_x', DATA_DIR / f'stress_x_polymer_{sim_name}.dat'),
    ('solvent_x', DATA_DIR / f'stress_x_solvent_{sim_name}.dat'),
    ('polymer_y', DATA_DIR / f'stress_y_polymer_{sim_name}.dat'),
    ('solvent_y', DATA_DIR / f'stress_y_solvent_{sim_name}.dat'),
]:
    print(f'  Reading {key} ...')
    stress_data[key] = read_ave_time_file(fp)

n_stress = len(stress_data['polymer_z'])
print(f'Stress snapshots: {n_stress}')

# Extract time series
all_timesteps  = [stress_data['polymer_z'][i][0] for i in range(n_stress)]
all_sigma_p_zz = [stress_data['polymer_z'][i][2] for i in range(n_stress)]
all_sigma_s_zz = [stress_data['solvent_z'][i][2] for i in range(n_stress)]
all_sigma_p_xx = [stress_data['polymer_x'][i][2] for i in range(n_stress)]
all_sigma_s_xx = [stress_data['solvent_x'][i][2] for i in range(n_stress)]
all_sigma_p_yy = [stress_data['polymer_y'][i][2] for i in range(n_stress)]
all_sigma_s_yy = [stress_data['solvent_y'][i][2] for i in range(n_stress)]

# Spatial coordinates from bin indices (bin k -> center at k*binWidth - binWidth/2)
bins_z   = stress_data['polymer_z'][0][1]
bins_x   = stress_data['polymer_x'][0][1]
bins_y   = stress_data['polymer_y'][0][1]
z_coords = bins_z * binWidth - binWidth / 2
x_coords = bins_x * binWidth - binWidth / 2
y_coords = bins_y * binWidth - binWidth / 2

# Normalised coordinates [0, 1] over each axis's bin span
z_norm = (z_coords - z_coords.min()) / (z_coords.max() - z_coords.min())
x_norm = (x_coords - x_coords.min()) / (x_coords.max() - x_coords.min())
y_norm = (y_coords - y_coords.min()) / (y_coords.max() - y_coords.min())

# ── Gel strain ────────────────────────────────────────────────────────────
ts_strain, L_arr, dL_arr = read_strain_file(DATA_DIR / f'strain_zz_{sim_name}.dat')
L_final   = float(L_arr[-1])
dL_final  = float(dL_arr[-1])
eps_final = dL_final / L_final
eps_arr   = dL_arr / L_arr
print(f'\nFinal: L={L_final:.4f}  ΔL={dL_final:.4f}  ε={eps_final:.4f}')

# ── Piston force ──────────────────────────────────────────────────────────
pf_data  = read_print_file(DATA_DIR / f'piston_force_{sim_name}.dat',
                            col_names=['step', 'F_piston_z'])
steps_pf = pf_data['step'].astype(int)
F_piston = pf_data['F_piston_z']
print(f'Piston force snapshots: {len(steps_pf)}  (t={steps_pf[0]} -> {steps_pf[-1]})')

# ── Box dimensions -> piston area ─────────────────────────────────────────
box_dims_file = DATA_DIR / f'box_dimensions_{sim_name}.dat'
if box_dims_file.exists():
    bd = read_print_file(box_dims_file, col_names=['step', 'lx', 'ly', 'lz'])
    piston_area = float(np.mean(bd['lx'])) * float(np.mean(bd['ly']))
    print(f'Piston area = {piston_area:.2f} sigma^2  '
          f'(lx={np.mean(bd["lx"]):.2f}, ly={np.mean(bd["ly"]):.2f})')
elif piston_area is None:
    raise ValueError(
        f'box_dimensions file not found at {box_dims_file} and piston_area '
        'not set in config. Set piston_area manually in the Config cell.')
else:
    print(f'Box dims file not found -- using config piston_area = {piston_area:.2f}')

# ── Displacement profiles (fix ave/chunk) ─────────────────────────────────
# Columns: [0]=chunk_id  [1]=Coord1(z, sigma)  [2]=Ncount  [3]=mean_uz
disp_snapshots = read_ave_chunk_file(DATA_DIR / f'disp_z_polymer_{sim_name}.dat')
n_disp  = len(disp_snapshots)
disp_ts = np.array([s[0] for s in disp_snapshots])
disp_z_raw  = disp_snapshots[0][1][:, 1]                          # (n_bins,)
disp_Ncount = np.array([s[1][:, 2] for s in disp_snapshots])      # (n_disp, n_bins)
disp_uz     = np.array([s[1][:, 3] for s in disp_snapshots])      # (n_disp, n_bins)
print(f'\nDisplacement snapshots: {n_disp}  (t={disp_ts[0]} -> {disp_ts[-1]})')

print(f'\nTimesteps: {all_timesteps[0]} -> {all_timesteps[-1]}')
print('Done.')

# ── True component z-binned stresses (σ_zz, σ_xx, σ_yy per species) ────────
# Written by slab_with_flow.lmp (fix avg_sigmazz/sigmaxx/sigmayy outputs).
# Used in the poroelastic decomposition:
#   σ'_comp(z) = σ_{p,comp}(z) + φ_p(z) · p_p(z),   p_p = σ_{s,comp}(z) / φ_s(z)
# Falls back to the isotropic z-profiles with a warning if files are absent
# (e.g. for simulation runs predating these outputs).
_comp_files = [
    ('polymer_zz',   DATA_DIR / f'sigmazz_polymer_{sim_name}.dat'),
    ('solvent_zz',   DATA_DIR / f'sigmazz_solvent_{sim_name}.dat'),
    ('polymer_xx_z', DATA_DIR / f'sigmaxx_polymer_{sim_name}.dat'),
    ('solvent_xx_z', DATA_DIR / f'sigmaxx_solvent_{sim_name}.dat'),
    ('polymer_yy_z', DATA_DIR / f'sigmayy_polymer_{sim_name}.dat'),
    ('solvent_yy_z', DATA_DIR / f'sigmayy_solvent_{sim_name}.dat'),
]
has_component_stresses = all(fp.exists() for _, fp in _comp_files)

if has_component_stresses:
    for key, fp in _comp_files:
        stress_data[key] = read_ave_time_file(fp)
    all_sigma_p_zz_comp = [stress_data['polymer_zz'][i][2]   for i in range(n_stress)]
    all_sigma_s_zz_comp = [stress_data['solvent_zz'][i][2]   for i in range(n_stress)]
    all_sigma_p_xx_comp = [stress_data['polymer_xx_z'][i][2] for i in range(n_stress)]
    all_sigma_s_xx_comp = [stress_data['solvent_xx_z'][i][2] for i in range(n_stress)]
    all_sigma_p_yy_comp = [stress_data['polymer_yy_z'][i][2] for i in range(n_stress)]
    all_sigma_s_yy_comp = [stress_data['solvent_yy_z'][i][2] for i in range(n_stress)]
    print('Component z-binned stresses loaded (σ_zz, σ_xx, σ_yy)')
else:
    # Placeholder: fall back to isotropic z-profiles
    all_sigma_p_zz_comp = all_sigma_p_zz
    all_sigma_s_zz_comp = all_sigma_s_zz
    all_sigma_p_xx_comp = all_sigma_p_zz   # wrong direction but avoids crash
    all_sigma_s_xx_comp = all_sigma_s_zz
    all_sigma_p_yy_comp = all_sigma_p_zz
    all_sigma_s_yy_comp = all_sigma_s_zz
    print('WARNING: component z-binned stress files not found — '
          'rerun slab_with_flow.lmp with σ_zz/σ_xx/σ_yy outputs enabled. '
          'Isotropic z-profiles used as placeholder; poroelastic results will be approximate.')

# ── Optional: dedicated ε=0 reference-state stresses (single snapshot each) ─
# Written by slab_with_flow.lmp's "Phase 2 pre-roll" block (added 2026-05).
# These are recorded while the piston is held at v=0 just before the real
# compression begins, giving a clean ε=0 reference for σ′.  When absent
# (older runs) we fall back to using stress index 0 of the main files,
# which is at ε ≈ comp_percent rather than ε = 0 — the analysis cell prints
# a warning in that case and the modulus may come out wrong.
_ref_files_main = [
    ('polymer_zz_ref',   DATA_DIR / f'sigmazz_polymer_ref_{sim_name}.dat'),
    ('solvent_zz_ref',   DATA_DIR / f'sigmazz_solvent_ref_{sim_name}.dat'),
    ('polymer_xx_z_ref', DATA_DIR / f'sigmaxx_polymer_ref_{sim_name}.dat'),
    ('solvent_xx_z_ref', DATA_DIR / f'sigmaxx_solvent_ref_{sim_name}.dat'),
    ('polymer_yy_z_ref', DATA_DIR / f'sigmayy_polymer_ref_{sim_name}.dat'),
    ('solvent_yy_z_ref', DATA_DIR / f'sigmayy_solvent_ref_{sim_name}.dat'),
]
has_ref_stresses = all(fp.exists() for _, fp in _ref_files_main)
if has_ref_stresses:
    for key, fp in _ref_files_main:
        stress_data[key] = read_ave_time_file(fp)
    # Each file has exactly one snapshot (the ε=0 reference)
    sig_p_zz_ref_dedicated = stress_data['polymer_zz_ref'][0][2]
    sig_s_zz_ref_dedicated = stress_data['solvent_zz_ref'][0][2]
    sig_p_xx_ref_dedicated = stress_data['polymer_xx_z_ref'][0][2]
    sig_s_xx_ref_dedicated = stress_data['solvent_xx_z_ref'][0][2]
    sig_p_yy_ref_dedicated = stress_data['polymer_yy_z_ref'][0][2]
    sig_s_yy_ref_dedicated = stress_data['solvent_yy_z_ref'][0][2]
    print('Dedicated ε=0 reference-state stresses loaded.')
else:
    print('NOTE: no dedicated ε=0 reference stress files found — fallback to '
          'stress index 0 of the regular files.  WARNING: that snapshot is at '
          'ε ≈ comp_percent, not ε = 0, so σ′_ref ≠ σ′(ε=0) and the resulting '
          'M can be wrong (even negative).  Rerun slab_with_flow.lmp with the '
          'reference-state recording enabled to fix.')


# all_sigma_s_zz_ss is stored as a 2-element list [ref_profile, final_profile]
# so the rest of the notebook (which uses index [0]=reference, [-1]=final)
# continues to work without modification.
has_ss_stresses = False
if PAIRS_FILE.exists() and TRAJ_FILE.exists():
    print('Computing sigma_s,ss from pair/local dump ...')
    _ss = compute_ss_stress_from_pairs(PAIRS_FILE, TRAJ_STRESS_FILE, binWidth)
    if _ss['z_bins'] is not None and len(_ss['timesteps']) >= 1:
        # ── Kinetic correction ─────────────────────────────────────────────
        # pair/local dumps carry only the pair-virial part of the solvent
        # stress.  In LAMMPS' (sign-flipped) convention, the missing kinetic
        # contribution on each diagonal component is +n_s(z) * kT per bin.
        # Add it now so downstream consumers see the *full* solvent self-
        # stress σ_ss + σ_kin,s (still without the ½σ_ps cross-term).
        _kin = _ss['n_s_z'] * kT                                # (n_frames, n_bins)
        _ss['sigma_ss_zz'] = _ss['sigma_ss_zz'] + _kin
        _ss['sigma_ss_xx'] = _ss['sigma_ss_xx'] + _kin
        _ss['sigma_ss_yy'] = _ss['sigma_ss_yy'] + _kin
        print(f'  Kinetic correction applied: +n_s(z)*kT  (kT = {kT:.3f}, '
              f'<n_s>_bulk-like max bin = {float(_ss["n_s_z"].max()):.3f})')

        from scipy.interpolate import interp1d as _i1d
        def _ss_interp(arr):
            return _i1d(_ss['z_bins'], arr, bounds_error=False, fill_value=0.0)(z_coords)
        n_ss  = len(_ss['timesteps'])
        n_avg = min(3, n_ss)
        # Reference = frame 0 of the main pairs dump.
        # Because pairlocal_dump fires at multiples of nfreq_stress AND the
        # pre-roll ends at a multiple of nfreq_stress (epoch-alignment fix),
        # frame 0 is captured while the piston is still frozen at v=0 — a
        # genuine ε=0 reference.  pairs_ref_*.dump (written one step earlier
        # in the pre-roll) is kept as belt-and-suspenders and overrides this
        # below when available, but the two should be nearly identical.
        _ref_zz = _ss_interp(_ss['sigma_ss_zz'][0])
        _ref_xx = _ss_interp(_ss['sigma_ss_xx'][0])
        _ref_yy = _ss_interp(_ss['sigma_ss_yy'][0])
        # Final = average of last n_avg frames (relaxed state)
        _fin_zz = _ss_interp(_ss['sigma_ss_zz'][-n_avg:].mean(0))
        _fin_xx = _ss_interp(_ss['sigma_ss_xx'][-n_avg:].mean(0))
        _fin_yy = _ss_interp(_ss['sigma_ss_yy'][-n_avg:].mean(0))
        all_sigma_s_zz_ss = [_ref_zz, _fin_zz]
        all_sigma_s_xx_ss = [_ref_xx, _fin_xx]
        all_sigma_s_yy_ss = [_ref_yy, _fin_yy]
        # Full time series — one profile per production pairs-dump frame.
        # Stored as a list parallel to all_sigma_p_zz / all_sigma_s_zz so
        # the evolution plot in Step 2b can overlay all three species.
        ss_ts_zz = [_ss_interp(_ss['sigma_ss_zz'][i]) for i in range(n_ss)]
        ss_ts_xx = [_ss_interp(_ss['sigma_ss_xx'][i]) for i in range(n_ss)]
        ss_ts_yy = [_ss_interp(_ss['sigma_ss_yy'][i]) for i in range(n_ss)]
        ss_ts_timesteps = list(_ss['timesteps'])
        has_ss_stresses = True
        print(f'  sigma_s,ss ready: {n_ss} frames, ref=frame 0 (ε=0, pre-roll end), final=avg of last {n_avg}')
    else:
        print('  WARNING: pairs dump was empty or unreadable')
else:
    print(f'NOTE: pairs dump not found at {PAIRS_FILE}')
    print('  Provide pairs_*.dump alongside other output files to enable ss-only stress.')

if not has_ss_stresses:
    all_sigma_s_zz_ss = all_sigma_s_zz_comp
    all_sigma_s_xx_ss = all_sigma_s_xx_comp
    all_sigma_s_yy_ss = all_sigma_s_yy_comp
    ss_ts_zz = []   # empty — no pairs dump available, evolution plot uses group-based fallback
    ss_ts_xx = []
    ss_ts_yy = []
    ss_ts_timesteps = []
    print('  Falling back to group-based sigma_s (includes 1/2 polymer-solvent virial).')
    print('  Reference subtraction will absorb the resulting baseline offset.')

# ── Replace contaminated ss reference with the ε=0 pairs_ref dump ─────────
# pairs_ref_*.dump + traj_ref_*.lammpstrj are written during the LAMMPS
# pre-roll (piston frozen at v=0), giving a true ε=0 reference for σ_s,ss.
# When these files are present they override the first-frame-of-PAIRS_FILE
# placeholder set above (which is a post-compression frame and should not
# be used for reference subtraction).
if has_ss_stresses and PAIRS_REF_FILE.exists() and TRAJ_REF_FILE.exists():
    print('Loading ε=0 pairs reference dump (pairs_ref_*.dump + traj_ref_*.lammpstrj)...')
    _ss_ref = compute_ss_stress_from_pairs(PAIRS_REF_FILE, TRAJ_REF_FILE, binWidth)
    if _ss_ref['z_bins'] is not None and len(_ss_ref['timesteps']) >= 1:
        # Mirror the kinetic correction applied to the main dump above so
        # both the reference and final ss profiles include +n_s*kT.
        _kin_ref = _ss_ref['n_s_z'] * kT
        _ss_ref['sigma_ss_zz'] = _ss_ref['sigma_ss_zz'] + _kin_ref
        _ss_ref['sigma_ss_xx'] = _ss_ref['sigma_ss_xx'] + _kin_ref
        _ss_ref['sigma_ss_yy'] = _ss_ref['sigma_ss_yy'] + _kin_ref

        def _ss_ref_interp(arr):
            return _i1d(_ss_ref['z_bins'], arr, bounds_error=False, fill_value=0.0)(z_coords)
        # Average all frames in the ref dump (typically just 1) → ε=0 profile
        all_sigma_s_zz_ss[0] = _ss_ref_interp(_ss_ref['sigma_ss_zz'].mean(axis=0))
        all_sigma_s_xx_ss[0] = _ss_ref_interp(_ss_ref['sigma_ss_xx'].mean(axis=0))
        all_sigma_s_yy_ss[0] = _ss_ref_interp(_ss_ref['sigma_ss_yy'].mean(axis=0))
        print(f'  σ_s,ss reference replaced with ε=0 pairs_ref dump '
              f'({len(_ss_ref["timesteps"])} frame(s) averaged).')
    else:
        print('  WARNING: pairs_ref dump found but empty/unreadable — '
              'keeping post-compression first-frame reference (results will be approximate).')
elif has_ss_stresses:
    # Main pairs dump loaded but no dedicated ref dump — warn clearly.
    missing = []
    if not PAIRS_REF_FILE.exists(): missing.append(f'pairs_ref  → {PAIRS_REF_FILE}')
    if not TRAJ_REF_FILE.exists():  missing.append(f'traj_ref   → {TRAJ_REF_FILE}')
    print('WARNING: dedicated ε=0 ss reference not available. Missing files:')
    for m in missing:
        print(f'    {m}')
    print('  σ_s,ss reference is the first frame of the main pairs dump (post-compression).')
    print('  Δσ_s,ss and downstream p_p / σ\'/ M will be inaccurate.')
    print('  Rerun slab_with_flow.lmp (updated pre-roll adds pairs_ref + traj_ref dumps).')

# ── Chemical potential (cavity-Widom) ─────────────────────────────────────
N_AVG_FRAMES = 4   # most-recent frames used for p_p / mu_total mean
has_mu_data  = False
mu_frames    = {}  # timestep -> (z_centers, mu_ex_array); filled below
if MU_SUMMARY_FILE.exists():
    try:
        mu_summary = read_mu_summary(MU_SUMMARY_FILE)
        has_mu_data = True
        print(f'Chemical potential summary loaded: '
              f'{len(mu_summary["z_center"])} bins, '
              f'mu_ex range [{float(np.nanmin(mu_summary["mu_ex_mean"])):.3f}, '
              f'{float(np.nanmax(mu_summary["mu_ex_mean"])):.3f}] eps')
    except Exception as _e:
        print(f'WARNING: could not read {MU_SUMMARY_FILE}: {_e}')
else:
    print(f'NOTE: no chemical potential data found at {MU_SUMMARY_FILE}')
    print('  Run cavity_widom.py on the trajectory to generate mu_z_cavity_summary file.')

# ── Per-frame mu_ex + last-N mean (shared by p_p calc and mu_total plot) ──
# Computed once here so cell 14 (p_p) and cell 17 (plot) are consistent.
if has_mu_data:
    if MU_FRAME_FILE.exists():
        _rows = {}
        with open(MU_FRAME_FILE) as _f:
            for _line in _f:
                _line = _line.strip()
                if not _line or _line.startswith('#'): continue
                _p = _line.split()
                if len(_p) < 8: continue
                _step = int(_p[0])
                _zcen = float(_p[1])
                _muv  = np.nan if _p[7] == 'nan' else float(_p[7])
                _rows.setdefault(_step, []).append((_zcen, _muv))
        for _step, _entries in _rows.items():
            _entries.sort(key=lambda x: x[0])
            mu_frames[_step] = (
                np.array([e[0] for e in _entries]),
                np.array([e[1] for e in _entries]),
            )
        print(f'Per-frame mu_ex loaded: {len(mu_frames)} frames from {MU_FRAME_FILE.name}')
    else:
        print(f'NOTE: {MU_FRAME_FILE.name} not found -- using all-frame summary mean for p_p')

    # Last-N-frame mean on the summary z-grid
    _z_mu      = mu_summary['z_center']
    _steps_all = sorted(mu_frames.keys())
    if _steps_all:
        _n_use      = min(N_AVG_FRAMES, len(_steps_all))
        _last_steps = _steps_all[-_n_use:]
        _stack = []
        for _s in _last_steps:
            _zf, _muf = mu_frames[_s]
            _stack.append(np.interp(_z_mu, _zf, _muf, left=np.nan, right=np.nan))
        _stack = np.array(_stack)
        import warnings as _w
        with _w.catch_warnings():
            _w.simplefilter('ignore', RuntimeWarning)
            mu_summary['mu_ex_lastN'] = np.nanmean(_stack, axis=0)
            if _n_use >= 2:
                _n_ok = np.sum(~np.isnan(_stack), axis=0)
                _std  = np.nanstd(_stack, axis=0, ddof=1)
                mu_summary['mu_se_lastN'] = np.where(
                    _n_ok >= 2, _std / np.sqrt(np.maximum(_n_ok, 1)), np.nan)
            else:
                mu_summary['mu_se_lastN'] = np.full_like(mu_summary['mu_ex_lastN'], np.nan)
        print(f'Last-{_n_use}-frame mu_ex mean computed (steps {_last_steps})')
    else:
        # No frame file -- fall back to all-frame summary mean
        mu_summary['mu_ex_lastN'] = mu_summary['mu_ex_mean']
        mu_summary['mu_se_lastN'] = mu_summary['mu_ex_stderr']

## Step 2: Partial Stress Evolution

Equilibration check: zz, xx, and yy partial stresses for polymer and solvent across all snapshots, colored early (dark) to late (bright). Convergence of $\sigma_{p,zz}$ and $\sigma_{s,zz}$ is the primary indicator of a relaxed final state.

In [ ]:
colors = plt.cm.viridis(np.linspace(0, 1, n_stress))
alpha  = 0.7

# ── Figure 1: zz components ───────────────────────────────────────────────
fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
fig1.suptitle(
    f'Equilibration Check (zz): {sim_name}\n{n_stress} snapshots',
    fontsize=12, fontweight='bold')
ax_pzz, ax_szz, ax_tzz = axes1
for i in range(n_stress):
    ax_pzz.plot(z_norm, all_sigma_p_zz[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_pzz.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_pzz.set(xlabel='$z/L_z$', ylabel=r'$\sigma_{p,zz}$',
           title=r'(a) Polymer $\sigma_{zz}$(z,t)', xlim=(0,1))
ax_pzz.grid(alpha=0.3)
for i in range(n_stress):
    ax_szz.plot(z_norm, all_sigma_s_zz[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_szz.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_szz.set(xlabel='$z/L_z$', ylabel=r'$\sigma_{s,zz}$',
           title=r'(b) Solvent $\sigma_{zz}$(z,t)', xlim=(0,1))
ax_szz.grid(alpha=0.3)
for i in range(n_stress):
    ax_tzz.plot(z_norm, all_sigma_p_zz[i] + all_sigma_s_zz[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_tzz.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_tzz.set(xlabel='$z/L_z$', ylabel=r'$\sigma_{p,zz} + \sigma_{s,zz}$',
           title=r'(c) Total $\sigma_{zz}$(z,t)', xlim=(0,1))
ax_tzz.grid(alpha=0.3)
sm = plt.cm.ScalarMappable(cmap='viridis',
                            norm=Normalize(vmin=all_timesteps[0], vmax=all_timesteps[-1]))
sm.set_array([])
fig1.colorbar(sm, ax=[ax_pzz, ax_szz, ax_tzz], fraction=0.03, pad=0.04).set_label('Timestep')
out1 = PLOT_DIR / f'equilibration_check_zz_{sim_name}.png'
plt.savefig(out1, dpi=150, bbox_inches='tight')
print(f'Saved: {out1}')
plt.show()
# ── Final-frame averages inside / outside gel ────────────────────────────
_sp = np.array(all_sigma_p_zz[-1])
_ss = np.array(all_sigma_s_zz[-1])
_st = _sp + _ss

# Gel interior: bins where polymer carries non-negligible stress
_gel_thresh = 0.05
_in  = _sp > _gel_thresh
_out = ~_in

print(f'\nFinal snapshot (t = {all_timesteps[-1]:.0f}) — mean zz partial stresses:')
print(f'  {"Region":<24}  {"σ_p,zz":>9}  {"σ_s,zz":>9}  {"σ_tot,zz":>10}  {"bins":>5}')
print(f'  {"-"*24}  {"-"*9}  {"-"*9}  {"-"*10}  {"-"*5}')
for _lbl, _mask in [('Inside gel', _in), ('Outside gel', _out)]:
    _sp_m = float(np.nanmean(_sp[_mask])) if _mask.any() else float('nan')
    _ss_m = float(np.nanmean(_ss[_mask])) if _mask.any() else float('nan')
    _st_m = float(np.nanmean(_st[_mask])) if _mask.any() else float('nan')
    print(f'  {_lbl:<24}  {_sp_m:>9.4f}  {_ss_m:>9.4f}  {_st_m:>10.4f}  {np.sum(_mask):>5d}')
print(f'  (gel interior: σ_p,zz > {_gel_thresh};  '
      f'inside {np.sum(_in)} bins, outside {np.sum(_out)} bins)')

# ── Figure 2: xx and yy components ───────────────────────────────────────
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
fig2.suptitle(
    f'Equilibration Check (xx/yy): {sim_name}\n{n_stress} snapshots',
    fontsize=12, fontweight='bold')
ax_pxx, ax_sxx, ax_txx = axes2[0]
ax_pyy, ax_syy, ax_tyy = axes2[1]
for i in range(n_stress):
    ax_pxx.plot(x_norm, all_sigma_p_xx[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_pxx.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_pxx.set(xlabel='$x/L_x$', ylabel=r'$\sigma_{p,xx}$',
           title=r'(a) Polymer $\sigma_{xx}$(x,t)', xlim=(0,1))
ax_pxx.grid(alpha=0.3)
for i in range(n_stress):
    ax_sxx.plot(x_norm, all_sigma_s_xx[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_sxx.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_sxx.set(xlabel='$x/L_x$', ylabel=r'$\sigma_{s,xx}$',
           title=r'(b) Solvent $\sigma_{xx}$(x,t)', xlim=(0,1))
ax_sxx.grid(alpha=0.3)
for i in range(n_stress):
    ax_txx.plot(x_norm, all_sigma_p_xx[i] + all_sigma_s_xx[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_txx.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_txx.set(xlabel='$x/L_x$', ylabel=r'$\sigma_{p,xx} + \sigma_{s,xx}$',
           title=r'(c) Total $\sigma_{xx}$(x,t)', xlim=(0,1))
ax_txx.grid(alpha=0.3)
for i in range(n_stress):
    ax_pyy.plot(y_norm, all_sigma_p_yy[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_pyy.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_pyy.set(xlabel='$y/L_y$', ylabel=r'$\sigma_{p,yy}$',
           title=r'(d) Polymer $\sigma_{yy}$(y,t)', xlim=(0,1))
ax_pyy.grid(alpha=0.3)
for i in range(n_stress):
    ax_syy.plot(y_norm, all_sigma_s_yy[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_syy.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_syy.set(xlabel='$y/L_y$', ylabel=r'$\sigma_{s,yy}$',
           title=r'(e) Solvent $\sigma_{yy}$(y,t)', xlim=(0,1))
ax_syy.grid(alpha=0.3)
for i in range(n_stress):
    ax_tyy.plot(y_norm, all_sigma_p_yy[i] + all_sigma_s_yy[i], '-', color=colors[i], lw=1.5, alpha=alpha)
ax_tyy.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_tyy.set(xlabel='$y/L_y$', ylabel=r'$\sigma_{p,yy} + \sigma_{s,yy}$',
           title=r'(f) Total $\sigma_{yy}$(y,t)', xlim=(0,1))
ax_tyy.grid(alpha=0.3)
sm2 = plt.cm.ScalarMappable(cmap='viridis',
                              norm=Normalize(vmin=all_timesteps[0], vmax=all_timesteps[-1]))
sm2.set_array([])
fig2.colorbar(sm2, ax=axes2, fraction=0.046, pad=0.04).set_label('Timestep')
out2 = PLOT_DIR / f'equilibration_check_xxyy_{sim_name}.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
print(f'Saved: {out2}')
plt.show()

## Step 2b: Solvent-Only ($\sigma_{s,ss}$) Stress Profiles

Per-component ($zz$, $xx$, $yy$) z-profiles of the **solvent–solvent-only virial stress** $\sigma_{s,ss}$, shown for the reference ($\varepsilon = 0$) and final ($\varepsilon \approx$ `comp_percent`) states.

---

### Why $\sigma_{s,ss}$ and not the group-based $\sigma_s$?

LAMMPS computes per-atom stress using the standard **viral theorem**.  When you request the stress of a *group* (e.g. the solvent group), LAMMPS assigns each pairwise interaction to the two participating atoms using the *equal-split* convention: each atom gets half the virial.  For a pair whose two atoms belong to *different* groups — e.g. a polymer atom (type 1 or 2) and a solvent atom (type 3) — **each atom gets ½ of the cross-pair virial**.  The solvent group therefore accumulates

$$
\sigma_s = \underbrace{\sigma_{s,ss}}_{\text{solvent–solvent}} + \underbrace{\tfrac{1}{2}\,\sigma_{ps}}_{\text{half polymer–solvent cross-term}}
$$

For **pore pressure** we want only the thermodynamic fluid pressure of the solvent, which comes from solvent–solvent interactions alone.  Using $\sigma_s$ instead of $\sigma_{s,ss}$ inflates $p_p$ by the cross-term and can distort the network stress and modulus.

---

### How $\sigma_{s,ss}$ is measured: the `pairs_*.dump` file

During the simulation `slab_with_flow.lmp` runs

```lammps
compute  pair_local  all  pair/local  fx  fy  fz
dump     pairs  all  local  <freq>  pairs_<sim_name>.dump  &
             c_ids_local[1]  c_ids_local[2]  &   # atom IDs
             c_ids_local[3]  c_ids_local[4]  &   # atom types
             c_pair_local[1] c_pair_local[2] c_pair_local[3]  # fx, fy, fz
```

`compute pair/local` records, for every interacting pair $(I, J)$, the force **on atom $I$ due to atom $J$** — one row per pair per snapshot.

---

### Reconstructing the stress tensor from pair forces

For each **solvent–solvent pair** (both types $= 3$) the notebook computes the Irving–Kirkwood virial contribution

$$
W_{\alpha\beta}^{IJ} = -\,r_{\alpha}^{IJ}\,f_{\beta}^{IJ}
$$

where $\mathbf{r}^{IJ} = \mathbf{r}_J - \mathbf{r}_I$ (PBC-minimum-image corrected) and $\mathbf{f}^{IJ}$ is the force on $I$ from $J$.  The sign follows the **LAMMPS pressure convention** (positive = compressive).

The pair is then assigned to the z-bin that contains its **midpoint**:

$$
z_\text{mid} = z_I + \tfrac{1}{2}\,r_z^{IJ}
$$

All virials in bin $k$ are summed and normalised by the bin volume $V_k = l_x\,l_y\,\Delta z$:

$$
\sigma_{s,ss,\alpha\beta}(z_k)
= \frac{1}{V_k}\sum_{\substack{(I,J)\,\text{ss pairs}\\ z_\text{mid}\in\text{bin}\,k}} W_{\alpha\beta}^{IJ}
$$

This is computed independently for $\alpha\beta \in \{zz,\,xx,\,yy\}$ and stored in `all_sigma_s_zz_ss`, `all_sigma_s_xx_ss`, `all_sigma_s_yy_ss` as a two-element list `[ref_profile, final_profile]`.

---

### Reference vs. final frames

The pairs dump is only recorded during the **relaxed final state** (not at $\varepsilon = 0$), so:

| State | σ_{s,ss} source |
|-------|----------------|
| **reference** | Frame 0 of `pairs_*.dump` — captured at the **epoch-end of the pre-roll**, while the piston is still frozen at v = 0 (genuine ε = 0). This works because `pairlocal_dump` fires at multiples of `nfreq_stress` and the pre-roll epoch-alignment fix ends the run at exactly such a multiple. |
| **reference (belt-and-suspenders)** | `pairs_ref_*.dump` — written one step earlier in the pre-roll via a `run 1` block. Practically identical to frame 0 of the main dump; overrides it when present. |
| **final** | average of last `n_avg` frames of `pairs_*.dump` (relaxed plateau) |

`pairs_*.dump` is paired with `traj_stress_*.lammpstrj` (atom positions at matching timesteps) so that $\mathbf{r}_{IJ}$ can be formed. Both dump at `nfreq_stress` — one frame per stress-profile snapshot.

When the pairs dumps are unavailable entirely, the notebook falls back to the **group-based $\sigma_s$** (which includes $\tfrac{1}{2}\sigma_{ps}$) — shown in the figure title.

---

### What to look for in the plots below

Two figures are produced:

**Figure A — time evolution** (requires `traj_stress_*.lammpstrj`): $\sigma_{s,ss,zz/xx/yy}(z,t)$ at every production snapshot, colored early→late by the same viridis colormap used in Step 2. This lets you overlay or directly compare the ss-only stress evolution against the partial polymer and solvent stresses.

**Figure B — reference vs final**: the two endpoint states with grey group-based $\sigma_s$ for comparison.
- $\sigma_{s,ss}$ (coloured) should sit **below** $\sigma_s$ (grey) everywhere inside the gel — the gap is $\tfrac{1}{2}\sigma_{ps}$.
- If the gap is negligible, the cross-term is small and the group-based $\sigma_s$ is a good approximation.
- The **reference–final difference** $\Delta\sigma_{s,ss}$ feeds directly into $\Delta p_p$ and from there into $\Delta\sigma'$ and the longitudinal modulus $M$.


In [ ]:
# == KINETIC CORRECTION TO SS-ONLY STRESS =================================
# sigma_s,ss from pair/local has NO kinetic contribution.  To compare
# apples-to-apples against the group-based partial stress (which includes
# kinetic + 1/2 sigma_ps), add the ideal-gas kinetic stress per component:
#
#     sigma_kin,s,ii(z) = n_s(z) * kT       (same for ii = xx, yy, zz)
#
# The kin-corrected stress  sigma_s,ss + sigma_kin,s  equals sigma_s,partial
# in the pure-solvent bulk slab (where 1/2 sigma_ps = 0).  Inside the gel
# the remaining gap between (ss+kin) and (group) is precisely 1/2 sigma_ps,
# so plotting both makes the polymer-solvent cross-term visible directly.
kT_lj = 1.0   # LJ-units temperature -- must match T_target in slab_with_flow.lmp

def _solvent_n_z(traj_file, frame_idx, bw, z_grid, solvent_type=3):
    """Solvent number density n_s(z) interpolated onto the stress z-grid."""
    _, _box, _atoms = read_lammpstrj_frame(traj_file, frame_idx)
    _a = np.asarray(_atoms)
    _types = _a[:, 1].astype(int)
    _sz = _a[_types == solvent_type, 5]
    _lx = _box['x'][1] - _box['x'][0]
    _ly = _box['y'][1] - _box['y'][0]
    _zlo, _zhi = _box['z']
    _vb = _lx * _ly * bw
    _edges = np.arange(_zlo, _zhi + bw, bw)
    _hist, _ = np.histogram(_sz, bins=_edges)
    _zc = 0.5 * (_edges[:-1] + _edges[1:])
    return interp1d(_zc, _hist / _vb,
                    bounds_error=False, fill_value=0.0)(z_grid)

print('Computing n_s(z) for kinetic correction ...')
n_s_ref_z = _solvent_n_z(TRAJ_FILE,  0, binWidth, z_coords)
n_s_fin_z = _solvent_n_z(TRAJ_FILE, -1, binWidth, z_coords)
sig_kin_s_ref_z = n_s_ref_z * kT_lj
sig_kin_s_fin_z = n_s_fin_z * kT_lj

# ss + kinetic -- mirror the [ref, final] structure of all_sigma_s_*_ss
sig_s_ref_ss_kin_zz   = np.array(all_sigma_s_zz_ss[0])  + sig_kin_s_ref_z
sig_s_ref_ss_kin_xx   = np.array(all_sigma_s_xx_ss[0])  + sig_kin_s_ref_z
sig_s_ref_ss_kin_yy   = np.array(all_sigma_s_yy_ss[0])  + sig_kin_s_ref_z
sig_s_final_ss_kin_zz = np.array(all_sigma_s_zz_ss[-1]) + sig_kin_s_fin_z
sig_s_final_ss_kin_xx = np.array(all_sigma_s_xx_ss[-1]) + sig_kin_s_fin_z
sig_s_final_ss_kin_yy = np.array(all_sigma_s_yy_ss[-1]) + sig_kin_s_fin_z

# group + kinetic -- LAMMPS group-based partial stress also uses
# compute stress/atom NULL (no kinetic), so to get the FULL Truesdell
# partial stress including kinetic we add n_s*kT here too.
if has_ref_stresses:
    _grp_ref_zz_kb = np.array(sig_s_zz_ref_dedicated)
    _grp_ref_xx_kb = np.array(sig_s_xx_ref_dedicated)
    _grp_ref_yy_kb = np.array(sig_s_yy_ref_dedicated)
else:
    _grp_ref_zz_kb = np.array(all_sigma_s_zz_comp[0])
    _grp_ref_xx_kb = np.array(all_sigma_s_xx_comp[0])
    _grp_ref_yy_kb = np.array(all_sigma_s_yy_comp[0])
sig_s_ref_partial_kin_zz   = _grp_ref_zz_kb + sig_kin_s_ref_z
sig_s_ref_partial_kin_xx   = _grp_ref_xx_kb + sig_kin_s_ref_z
sig_s_ref_partial_kin_yy   = _grp_ref_yy_kb + sig_kin_s_ref_z
sig_s_final_partial_kin_zz = np.array(all_sigma_s_zz_comp[-1]) + sig_kin_s_fin_z
sig_s_final_partial_kin_xx = np.array(all_sigma_s_xx_comp[-1]) + sig_kin_s_fin_z
sig_s_final_partial_kin_yy = np.array(all_sigma_s_yy_comp[-1]) + sig_kin_s_fin_z

# Sanity prints -- gap between ss+kin and group should be ~0 in the bulk
# slab (no polymer there -> 1/2 sigma_ps = 0) and grow into the gel.
_bulk_mask = z_norm > 0.85
_grp_fin_bulk = float(np.nanmean(np.array(all_sigma_s_zz_comp[-1])[_bulk_mask]))
_kin_fin_bulk = float(np.nanmean(sig_s_final_ss_kin_zz[_bulk_mask]))
_ss_fin_bulk  = float(np.nanmean(np.array(all_sigma_s_zz_ss[-1])[_bulk_mask]))
print(f'  kT_lj = {kT_lj:.3f}')
print(f'  Bulk-slab (z/Lz > 0.85) zz averages (final):')
_grp_kin_fin_bulk = float(np.nanmean(sig_s_final_partial_kin_zz[_bulk_mask]))
print(f'    sigma_s,ss                  = {_ss_fin_bulk:.4f}   (LAMMPS pair/local, virial only)')
print(f'    sigma_s,partial (LAMMPS)    = {_grp_fin_bulk:.4f}   (group stress/atom NULL, virial only)')
print(f'    sigma_s,ss + n_s kT         = {_kin_fin_bulk:.4f}')
print(f'    sigma_s,partial + n_s kT    = {_grp_kin_fin_bulk:.4f}   (TRUE partial w/ kinetic)')
print(f'    (ss+kin) - (partial+kin)    = {_kin_fin_bulk - _grp_kin_fin_bulk:+.4f}   '
      f'(should be ~0 in pure-solvent slab: 1/2 sigma_ps = 0 there)')

# == Solvent-only stress profiles: time evolution + reference vs final ======
#
# Two figures, mirroring the structure of Step 2:
#
#   Figure A — evolution (all production frames, colored early→late)
#              Uses ss_ts_zz/xx/yy when the pairs dump was loaded at
#              nfreq_stress cadence.  Falls back to the 2-state
#              all_sigma_s_*_ss ref/final plot when ss_ts_* is empty.
#
#   Figure B — reference vs final overlay (2 states)
#              Compares σ_s,ss (pairs dump) against the group-based σ_s
#              (fix ave/time) to show the polymer-solvent cross-term gap.
#
# Source label reflects whether the true pairs dump or the group fallback
# was used.

src_label = ('pair/local dump (ss-only)' if has_ss_stresses
             else 'group-based sigma_s (fallback; includes polymer-solvent virial)')

# ── Figure A: time evolution ───────────────────────────────────────────────
has_full_ts = has_ss_stresses and len(ss_ts_zz) > 0

if has_full_ts:
    n_ss_ts = len(ss_ts_zz)
    colors_ss = plt.cm.viridis(np.linspace(0, 1, n_ss_ts))

    fig_A, axes_A = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    fig_A.suptitle(
        f'Equilibration Check — Solvent-Only σ_{{s,ss}}: {sim_name}\n'
        f'{n_ss_ts} snapshots  |  source: {src_label}',
        fontsize=12, fontweight='bold')

    comp_info = [
        (axes_A[0], ss_ts_zz, r'$\sigma_{s,ss,zz}$(z,t)', r'$\sigma_{s,ss,zz}$'),
        (axes_A[1], ss_ts_xx, r'$\sigma_{s,ss,xx}$(z,t)', r'$\sigma_{s,ss,xx}$'),
        (axes_A[2], ss_ts_yy, r'$\sigma_{s,ss,yy}$(z,t)', r'$\sigma_{s,ss,yy}$'),
    ]
    for ax, ts_list, title, ylabel in comp_info:
        for i, prof in enumerate(ts_list):
            ax.plot(z_norm, prof, '-', color=colors_ss[i], lw=1.5, alpha=0.7)
        ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
        ax.set(xlabel=r'$z/L_z$', ylabel=ylabel, title=title, xlim=(0, 1))
        ax.grid(alpha=0.3)

    sm_ss = plt.cm.ScalarMappable(cmap='viridis',
                                   norm=Normalize(vmin=ss_ts_timesteps[0],
                                                  vmax=ss_ts_timesteps[-1]))
    sm_ss.set_array([])
    fig_A.colorbar(sm_ss, ax=axes_A, fraction=0.03, pad=0.04).set_label('Timestep')
    out_A = PLOT_DIR / f'equilibration_check_ss_{sim_name}.png'
    plt.savefig(out_A, dpi=150, bbox_inches='tight')
    print(f'Saved: {out_A}')
    plt.show()
else:
    print('NOTE: ss_ts_* is empty (no pairs dump at nfreq_stress cadence).')
    print('  Skipping evolution plot.  Rerun with traj_stress_*.lammpstrj to enable.')

# ── Figure B: reference vs final + group-based comparison ─────────────────
sig_ss_ref_zz  = np.array(all_sigma_s_zz_ss[0])
sig_ss_ref_xx  = np.array(all_sigma_s_xx_ss[0])
sig_ss_ref_yy  = np.array(all_sigma_s_yy_ss[0])
sig_ss_fin_zz  = np.array(all_sigma_s_zz_ss[-1])
sig_ss_fin_xx  = np.array(all_sigma_s_xx_ss[-1])
sig_ss_fin_yy  = np.array(all_sigma_s_yy_ss[-1])

if has_ref_stresses:
    grp_ref_zz = np.array(sig_s_zz_ref_dedicated)
    grp_ref_xx = np.array(sig_s_xx_ref_dedicated)
    grp_ref_yy = np.array(sig_s_yy_ref_dedicated)
else:
    grp_ref_zz = np.array(all_sigma_s_zz_comp[0])
    grp_ref_xx = np.array(all_sigma_s_xx_comp[0])
    grp_ref_yy = np.array(all_sigma_s_yy_comp[0])
grp_fin_zz = np.array(all_sigma_s_zz_comp[-1])
grp_fin_xx = np.array(all_sigma_s_xx_comp[-1])
grp_fin_yy = np.array(all_sigma_s_yy_comp[-1])

comps = [
    (r'$\sigma_{s,ss,zz}$', sig_ss_ref_zz, sig_ss_fin_zz, grp_ref_zz, grp_fin_zz,
     sig_s_ref_ss_kin_zz, sig_s_final_ss_kin_zz),
    (r'$\sigma_{s,ss,xx}$', sig_ss_ref_xx, sig_ss_fin_xx, grp_ref_xx, grp_fin_xx,
     sig_s_ref_ss_kin_xx, sig_s_final_ss_kin_xx),
    (r'$\sigma_{s,ss,yy}$', sig_ss_ref_yy, sig_ss_fin_yy, grp_ref_yy, grp_fin_yy,
     sig_s_ref_ss_kin_yy, sig_s_final_ss_kin_yy),
]

fig_B, axes_B = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
fig_B.suptitle(
    f'Solvent-Only Stress: Reference vs Final   |   {RUN_ID}\n'
    f'Source: {src_label}',
    fontsize=13, fontweight='bold')

for ax, (label, ss_ref, ss_fin, grp_ref, grp_fin, kin_ref, kin_fin) in zip(axes_B, comps):
    ax.plot(z_norm, grp_ref, '-',  color='0.75', lw=1.2,
            label=r'$\sigma_s$ ref (group)', zorder=1)
    ax.plot(z_norm, grp_fin, '--', color='0.75', lw=1.2,
            label=r'$\sigma_s$ final (group)', zorder=1)
    ax.plot(z_norm, ss_ref,  '-',  color='steelblue', lw=2.2,
            label=r'$\sigma_{s,ss}$ ref', zorder=2)
    ax.plot(z_norm, ss_fin,  '--', color='crimson',   lw=2.2,
            label=r'$\sigma_{s,ss}$ final', zorder=2)
    ax.axhline(0, color='k', ls=':', lw=0.8, alpha=0.5)
    ax.set_xlabel(r'$z/L_z$')
    ax.set_ylabel(r'stress  (LJ)')
    ax.set_title(label, fontsize=16)
    ax.set_xlim(0, 1)
    ax.legend(fontsize=11, loc='best')
    ax.grid(alpha=0.3)

out_B = PLOT_DIR / f'sigma_ss_profiles_{sim_name}.png'
plt.savefig(out_B, dpi=150, bbox_inches='tight')
print(f'Saved: {out_B}')
plt.show()
# ── Figure C: GROUP-BASED partial-stress reference vs final ───────────────
# Mirror of Figure B but for the full Truesdell partial solvent stress
#   sigma_s,partial = sigma_ss + 1/2 sigma_ps + sigma_kin,s
# i.e. the same arrays (sig_s_*) that p_p uses when p_p_source = 'group'
# in cell 14.  Compare against Figure B above to see how much of the gel
# stress signal is carried by the cross-term + kinetic pieces that ss-only
# discards.
if has_ref_stresses:
    grp_ref_zz_c = np.array(sig_s_zz_ref_dedicated)
    grp_ref_xx_c = np.array(sig_s_xx_ref_dedicated)
    grp_ref_yy_c = np.array(sig_s_yy_ref_dedicated)
else:
    grp_ref_zz_c = np.array(all_sigma_s_zz_comp[0])
    grp_ref_xx_c = np.array(all_sigma_s_xx_comp[0])
    grp_ref_yy_c = np.array(all_sigma_s_yy_comp[0])
grp_fin_zz_c = np.array(all_sigma_s_zz_comp[-1])
grp_fin_xx_c = np.array(all_sigma_s_xx_comp[-1])
grp_fin_yy_c = np.array(all_sigma_s_yy_comp[-1])

fig_C, axes_C = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
fig_C.suptitle(
    f'Group-Based Partial Solvent Stress: Reference vs Final   |   {RUN_ID}\n'
    r'$\sigma_{s,\mathrm{partial}} = \sigma_{ss} + 1/2\sigma_{ps} + \sigma_{\mathrm{kin},s}$',
    fontsize=13, fontweight='bold')

comps_C = [
    (r'$\sigma_{s,\mathrm{partial},zz}$', grp_ref_zz_c, grp_fin_zz_c,
     sig_s_ref_ss_kin_zz, sig_s_final_ss_kin_zz,
     sig_s_ref_partial_kin_zz, sig_s_final_partial_kin_zz),
    (r'$\sigma_{s,\mathrm{partial},xx}$', grp_ref_xx_c, grp_fin_xx_c,
     sig_s_ref_ss_kin_xx, sig_s_final_ss_kin_xx,
     sig_s_ref_partial_kin_xx, sig_s_final_partial_kin_xx),
    (r'$\sigma_{s,\mathrm{partial},yy}$', grp_ref_yy_c, grp_fin_yy_c,
     sig_s_ref_ss_kin_yy, sig_s_final_ss_kin_yy,
     sig_s_ref_partial_kin_yy, sig_s_final_partial_kin_yy),
]

for ax, (label, ref, fin, kin_ref, kin_fin, pkin_ref, pkin_fin) in zip(axes_C, comps_C):
    # LAMMPS uses `compute stress/atom NULL` for BOTH stresses, so grp/ss
    # arrays are both pair-virial only.  To compare apples-to-apples we add
    # n_s*kT to both: in the pure-solvent slab (ss+kin) overlays (partial+kin)
    # because 1/2 sigma_ps = 0 there; inside the gel they diverge by 1/2 sigma_ps.
    ax.plot(z_norm, ref, '-',  color='steelblue', lw=2.2,
            label=r'$\sigma_{s,\mathrm{partial}}$ ref (LAMMPS, virial)', zorder=3)
    ax.plot(z_norm, fin, '--', color='crimson',   lw=2.2,
            label=r'$\sigma_{s,\mathrm{partial}}$ final (LAMMPS, virial)', zorder=3)
    ax.plot(z_norm, pkin_ref, '-',  color='navy',     lw=1.5, alpha=0.85,
            label=r'$\sigma_{s,\mathrm{partial}}+n_s kT$ ref', zorder=2)
    ax.plot(z_norm, pkin_fin, '--', color='darkred',  lw=1.5, alpha=0.85,
            label=r'$\sigma_{s,\mathrm{partial}}+n_s kT$ final', zorder=2)
    ax.plot(z_norm, kin_ref, '-',  color='darkgreen',  lw=1.5, alpha=0.85,
            label=r'$\sigma_{s,ss}+n_s kT$ ref', zorder=2)
    ax.plot(z_norm, kin_fin, '--', color='darkorange', lw=1.5, alpha=0.85,
            label=r'$\sigma_{s,ss}+n_s kT$ final', zorder=2)
    ax.axhline(0, color='k', ls=':', lw=0.8, alpha=0.5)
    ax.set_xlabel(r'$z/L_z$')
    ax.set_ylabel(r'stress  (LJ)')
    ax.set_title(label, fontsize=16)
    ax.set_xlim(0, 1)
    ax.legend(fontsize=12, loc='best')
    ax.grid(alpha=0.3)

out_C = PLOT_DIR / f'sigma_s_partial_profiles_{sim_name}.png'
plt.savefig(out_C, dpi=150, bbox_inches='tight')
print(f'Saved: {out_C}')
plt.show()



## Step 3: Final-State Profiles

Voronoi volume fractions computed once for the final trajectory frame. Used for the poroelastic decomposition: pore pressure $p_p(z)$, effective network stress $\sigma'_{zz}(z)$, and longitudinal modulus $M(z) = \sigma'_{zz}(z)/\varepsilon_{zz}$.

In [ ]:
from scipy.interpolate import interp1d

idx_final = -1  # last stress snapshot

# Use true component z-binned stresses (σ_zz, σ_xx, σ_yy per species).
# 'has_component_stresses' and the _comp arrays are set in the Load cell above.
sig_p_final    = np.array(all_sigma_p_zz_comp[idx_final])  # σ_{p,zz}(z)
sig_s_final    = np.array(all_sigma_s_zz_comp[idx_final])  # σ_{s,zz}(z)
sig_p_final_xx = np.array(all_sigma_p_xx_comp[idx_final])  # σ_{p,xx}(z)
sig_s_final_xx = np.array(all_sigma_s_xx_comp[idx_final])  # σ_{s,xx}(z)
sig_p_final_yy = np.array(all_sigma_p_yy_comp[idx_final])  # σ_{p,yy}(z)
sig_s_final_yy = np.array(all_sigma_s_yy_comp[idx_final])  # σ_{s,yy}(z)

# Solvent-solvent-only stress (preferred for p_p when available; falls back to
# group-based σ_s above — flagged in cell 7).
sig_s_final_ss_zz = np.array(all_sigma_s_zz_ss[idx_final])
sig_s_final_ss_xx = np.array(all_sigma_s_xx_ss[idx_final])
sig_s_final_ss_yy = np.array(all_sigma_s_yy_ss[idx_final])

# Reference (uncompressed) snapshot.
# Preferred source: dedicated "*_ref_*.dat" files written during the LAMMPS
# "Phase 2 pre-roll" block — these are recorded with the piston frozen at
# v=0, so ε = 0 exactly.
# Fallback: stress index 0 of the regular files.  Because the regular fix
# ave/time fixes only emit their first snapshot at t = nfreq_stress (well
# AFTER comp_percent has been reached and relaxation has begun), this fallback
# snapshot is at ε ≈ comp_percent, NOT ε = 0.  Using it produces a
# baseline that already contains most of the elastic stress, so Δσ′ shrinks
# (often to a small negative number).  The cell prints a warning in that
# case — please rerun slab_with_flow.lmp with the reference-state recording.
if has_ref_stresses:
    sig_p_ref     = np.array(sig_p_zz_ref_dedicated)
    sig_s_ref     = np.array(sig_s_zz_ref_dedicated)
    sig_p_ref_xx  = np.array(sig_p_xx_ref_dedicated)
    sig_s_ref_xx  = np.array(sig_s_xx_ref_dedicated)
    sig_p_ref_yy  = np.array(sig_p_yy_ref_dedicated)
    sig_s_ref_yy  = np.array(sig_s_yy_ref_dedicated)
    # ss-only ε=0 reference for pore pressure.
    # all_sigma_s_*_ss[0] is the first frame of the production pairs dump,
    # which is captured at the pre-roll epoch boundary (piston frozen, ε=0).
    # This is now preferred over the group-based dedicated ref files because
    # group-based σ_s includes ½σ_ps, inflating p_p_ref by ~2-3×  and making
    # Δp_p and M wrong.  Using the ss-only frame 0 ensures the reference and
    # final p_p are computed from the same stress source.
    if has_ss_stresses:
        sig_s_ref_ss_zz = np.array(all_sigma_s_zz_ss[0])
        sig_s_ref_ss_xx = np.array(all_sigma_s_xx_ss[0])
        sig_s_ref_ss_yy = np.array(all_sigma_s_yy_ss[0])
        print('  Pore-pressure reference: ss-only from pairs dump frame 0 (ε=0 pre-roll end).')
    else:
        # No pairs dump at all — group-based is the only option; reference
        # subtraction will partially cancel the ps cross-term contamination.
        sig_s_ref_ss_zz = np.array(sig_s_zz_ref_dedicated)
        sig_s_ref_ss_xx = np.array(sig_s_xx_ref_dedicated)
        sig_s_ref_ss_yy = np.array(sig_s_yy_ref_dedicated)
        print('  Pore-pressure reference: group-based σ_s (pairs dump unavailable).')
        print('  WARNING: ½σ_ps cross-term inflates p_p_ref. Δp_p and M will be approximate.')
    idx_ref = 'dedicated'
    print('Reference state: dedicated ε=0 recording (preferred).')
else:
    idx_ref = 0
    sig_p_ref     = np.array(all_sigma_p_zz_comp[idx_ref])
    sig_s_ref     = np.array(all_sigma_s_zz_comp[idx_ref])
    sig_p_ref_xx  = np.array(all_sigma_p_xx_comp[idx_ref])
    sig_s_ref_xx  = np.array(all_sigma_s_xx_comp[idx_ref])
    sig_p_ref_yy  = np.array(all_sigma_p_yy_comp[idx_ref])
    sig_s_ref_yy  = np.array(all_sigma_s_yy_comp[idx_ref])
    sig_s_ref_ss_zz = np.array(all_sigma_s_zz_ss[idx_ref])
    sig_s_ref_ss_xx = np.array(all_sigma_s_xx_ss[idx_ref])
    sig_s_ref_ss_yy = np.array(all_sigma_s_yy_ss[idx_ref])
    print('Reference state: WARNING — falling back to stress index 0, which '
          'is at ε ≈ comp_percent (NOT ε = 0).  Δσ′ and M will be unreliable. '
          'Rerun slab_with_flow.lmp with the Phase-2 pre-roll block to fix.')

if not has_component_stresses:
    print('NOTE: using isotropic placeholder — component stress values are approximate.')

# ── Voronoi volume fractions ─────────────────────────────────────────────
# We need φ at TWO states: the un-deformed reference (idx_ref) and the relaxed
# final state.  For the final state we time-average over the last
# `n_phi_avg` trajectory frames to suppress bin-to-bin Voronoi noise — the
# user reported visible bin scatter in single-frame Voronoi profiles.
n_phi_avg = 3   # number of trailing trajectory frames to average for "current" φ
ref_frame_idx = 0   # first trajectory frame = uncompressed reference

# Count traj frames so we can address the trailing ones by absolute index
def _count_traj_frames(filepath):
    n = 0
    with open(filepath) as f:
        for line in f:
            if 'ITEM: TIMESTEP' in line:
                n += 1
    return n

n_traj_frames = _count_traj_frames(TRAJ_FILE)
print(f'Trajectory contains {n_traj_frames} frames.')

# ── Reference frame Voronoi ─────────────────────────────────────────────
print(f'\nComputing Voronoi volume fractions for REFERENCE frame ({ref_frame_idx}, uncompressed)...')
_, box_ref, atoms_ref = read_lammpstrj_frame(TRAJ_FILE, ref_frame_idx)
box_bounds_ref = {'x': box_ref['x'], 'y': box_ref['y'], 'z': box_ref['z']}
z_vor_ref, phi_p_vor_ref, phi_s_vor_ref = compute_volume_fractions_1d_voronoi(
    atoms_ref, box_bounds_ref, binWidth, 'z')
zlo_r, zhi_r = box_ref['z']
Lz_r = zhi_r - zlo_r
z_vor_ref_norm = (z_vor_ref - zlo_r) / Lz_r

phi_p_ref = interp1d(z_vor_ref, phi_p_vor_ref, bounds_error=False, fill_value=0.0)(z_coords)
phi_s_ref = interp1d(z_vor_ref, phi_s_vor_ref, bounds_error=False, fill_value=0.0)(z_coords)

# ── Final-state Voronoi: time-averaged over last n_phi_avg traj frames ───
# Frames are at uniform stride, all in the relaxed plateau (piston force is
# already flat by frame -3 in your data). Averaging φ_p, φ_s per stress bin
# across frames knocks down the per-bin shot noise without smearing the
# spatial profile.  Final-frame box is used for normalisation since L_z is
# fixed after the piston halts.
print(f'\nComputing time-averaged Voronoi volume fractions over last {n_phi_avg} traj frames...')
phi_p_frames = []
phi_s_frames = []
for k in range(n_phi_avg):
    fi = n_traj_frames - n_phi_avg + k     # last n_phi_avg frames
    print(f'  Frame {fi} (of {n_traj_frames})...')
    _, box_k, atoms_k = read_lammpstrj_frame(TRAJ_FILE, fi)
    bb_k = {'x': box_k['x'], 'y': box_k['y'], 'z': box_k['z']}
    z_vor_k, phi_p_vor_k, phi_s_vor_k = compute_volume_fractions_1d_voronoi(
        atoms_k, bb_k, binWidth, 'z')
    phi_p_k = interp1d(z_vor_k, phi_p_vor_k, bounds_error=False, fill_value=0.0)(z_coords)
    phi_s_k = interp1d(z_vor_k, phi_s_vor_k, bounds_error=False, fill_value=0.0)(z_coords)
    phi_p_frames.append(phi_p_k)
    phi_s_frames.append(phi_s_k)

phi_p_frames = np.array(phi_p_frames)   # (n_phi_avg, n_z_bins)
phi_s_frames = np.array(phi_s_frames)
phi_p_final = phi_p_frames.mean(axis=0)
phi_s_final = phi_s_frames.mean(axis=0)

# Diagnostics on the time-averaging
gel_diag = phi_p_final > 0.1
phi_p_std_per_bin = phi_p_frames.std(axis=0, ddof=1) if n_phi_avg >= 2 else np.zeros_like(phi_p_final)
phi_s_std_per_bin = phi_s_frames.std(axis=0, ddof=1) if n_phi_avg >= 2 else np.zeros_like(phi_s_final)
print(f'  Mean per-bin std of φ_p across frames (gel): {np.nanmean(phi_p_std_per_bin[gel_diag]):.4f}')
print(f'  Mean per-bin std of φ_s across frames (gel): {np.nanmean(phi_s_std_per_bin[gel_diag]):.4f}')

# Keep a single-frame final Voronoi too (for the volume-fraction plot in cell 20
# which currently expects z_vor, phi_p_vor, phi_s_vor from THE final frame).
print(f'\nComputing Voronoi for single final frame (used by Plot A in next cell)...')
_, box_final, atoms_final = read_lammpstrj_frame(TRAJ_FILE, -1)
box_bounds_final = {'x': box_final['x'], 'y': box_final['y'], 'z': box_final['z']}
z_vor, phi_p_vor, phi_s_vor = compute_volume_fractions_1d_voronoi(
    atoms_final, box_bounds_final, binWidth, 'z')
zlo_f, zhi_f = box_final['z']
Lz_f         = zhi_f - zlo_f
z_vor_norm   = (z_vor - zlo_f) / Lz_f
# ==========================================================================
#   COUNT-BASED AND PACKING-FRACTION VOLUME FRACTIONS
#   (all three phi_s variants available for the method selector in cell 14)
# ==========================================================================

# ── LJ geometry (from slab_with_flow.lmp) ──────────────────────────────────
# sigma = 1.0 (all types), pair_style lj/cut 1.122 = WCA (r_cut = 2^(1/6) sigma)
sigma_lj = 1.0
d_eff    = 1.122                     # 2^(1/6)*sigma -- effective hard-core diameter
v_bead   = (np.pi / 6) * d_eff**3   # ~0.7396 sigma^3

# ── Count-based phi_s (N_s / N_total per bin) ──────────────────────────────
# Equal to Voronoi for monodisperse beads; included as an explicit sanity check.
z_cnt_ref, phi_p_cnt_ref, phi_s_cnt_ref = compute_volume_fractions_1d(
    atoms_ref,   box_bounds_ref,   binWidth, 'z')
z_cnt_fin, phi_p_cnt_fin, phi_s_cnt_fin = compute_volume_fractions_1d(
    atoms_final, box_bounds_final, binWidth, 'z')
# Interpolate onto the stress-profile grid (z_coords)
phi_s_cnt_ref_z = interp1d(z_cnt_ref, phi_s_cnt_ref, bounds_error=False, fill_value=0.0)(z_coords)
phi_s_cnt_fin_z = interp1d(z_cnt_fin, phi_s_cnt_fin, bounds_error=False, fill_value=0.0)(z_coords)
phi_p_cnt_ref_z = interp1d(z_cnt_ref, phi_p_cnt_ref, bounds_error=False, fill_value=0.0)(z_coords)
phi_p_cnt_fin_z = interp1d(z_cnt_fin, phi_p_cnt_fin, bounds_error=False, fill_value=0.0)(z_coords)

# ── Packing-fraction phi_s (rho_s * v_bead) ────────────────────────────────
def _packing_phi(atoms_data, box_bounds, bw, vb):
    # Per-z-bin packing fractions and number density from atom positions.
    a   = np.array(atoms_data)
    tp  = a[:, 1].astype(int)
    pz  = a[:, 3:6]
    pm  = (tp == 1) | (tp == 2)
    sm  = (tp == 3)
    lx  = box_bounds['x'][1] - box_bounds['x'][0]
    ly  = box_bounds['y'][1] - box_bounds['y'][0]
    zlo, zhi = box_bounds['z']
    V_b   = lx * ly * bw
    edges = np.arange(zlo, zhi + bw, bw)
    z_c   = (edges[:-1] + edges[1:]) / 2
    phi_p_out = np.zeros(len(z_c))
    phi_s_out = np.zeros(len(z_c))
    rho_t_out = np.zeros(len(z_c))
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (pz[:, 2] >= lo) & (pz[:, 2] < hi)
        phi_p_out[i] = np.sum(pm & mask) / V_b * vb
        phi_s_out[i] = np.sum(sm & mask) / V_b * vb
        rho_t_out[i] = np.sum((pm | sm) & mask) / V_b
    return z_c, phi_p_out, phi_s_out, rho_t_out

z_pk_ref, phi_p_pk_ref, phi_s_pk_ref, rho_t_pk_ref = _packing_phi(
    atoms_ref,   box_bounds_ref,   binWidth, v_bead)
z_pk_fin, phi_p_pk_fin, phi_s_pk_fin, rho_t_pk_fin = _packing_phi(
    atoms_final, box_bounds_final, binWidth, v_bead)

# Interpolate onto z_coords
phi_s_pk_ref_z = interp1d(z_pk_ref, phi_s_pk_ref, bounds_error=False, fill_value=0.0)(z_coords)
phi_s_pk_fin_z = interp1d(z_pk_fin, phi_s_pk_fin, bounds_error=False, fill_value=0.0)(z_coords)
phi_p_pk_ref_z = interp1d(z_pk_ref, phi_p_pk_ref, bounds_error=False, fill_value=0.0)(z_coords)
phi_p_pk_fin_z = interp1d(z_pk_fin, phi_p_pk_fin, bounds_error=False, fill_value=0.0)(z_coords)
rho_t_pk_fin_z = interp1d(z_pk_fin, rho_t_pk_fin, bounds_error=False, fill_value=0.0)(z_coords)

# ── Gel-interior density diagnostics ───────────────────────────────────────
_gel_d        = phi_p_final > phi_gel_threshold   # Voronoi gel mask on z_coords
rho_total_gel = float(np.nanmean(rho_t_pk_fin_z[_gel_d]))
rho_s_gel     = float(np.nanmean(phi_s_pk_fin_z[_gel_d])) / v_bead
rho_p_gel     = float(np.nanmean(phi_p_pk_fin_z[_gel_d])) / v_bead
print('Gel-interior density (final frame, mobile atoms only):')
print(f'  rho_total = {rho_total_gel:.4f}  sigma^-3')
print(f'  rho_p     = {rho_p_gel:.4f}  sigma^-3')
print(f'  rho_s     = {rho_s_gel:.4f}  sigma^-3')
print(f'  d_eff     = {d_eff:.4f}  sigma  (2^(1/6)*sigma, WCA hard-core)')
print(f'  v_bead    = {v_bead:.4f}  sigma^3  (pi/6 * d_eff^3)')
print(f'  phi_s (Voronoi)  gel mean = {float(np.nanmean(phi_s_final[_gel_d])):.4f}')
print(f'  phi_s (count)    gel mean = {float(np.nanmean(phi_s_cnt_fin_z[_gel_d])):.4f}')
print(f'  phi_s (packing)  gel mean = {float(np.nanmean(phi_s_pk_fin_z[_gel_d])):.4f}')


# ── Gel z-bounds for shading in all downstream plots ─────────────────────
_gel_zbool = phi_p_final > phi_gel_threshold
z_gel_lo = float(z_coords[_gel_zbool].min()) if _gel_zbool.any() else float(z_coords[0])
z_gel_hi = float(z_coords[_gel_zbool].max()) if _gel_zbool.any() else float(z_coords[-1])
z_gel_lo_norm = (z_gel_lo - zlo_f) / Lz_f
z_gel_hi_norm = (z_gel_hi - zlo_f) / Lz_f
print(f'Gel z-bounds: [{z_gel_lo:.2f}, {z_gel_hi:.2f}] sigma  '
      f'(norm: [{z_gel_lo_norm:.3f}, {z_gel_hi_norm:.3f}])')

In [ ]:
# ==========================================================================
#   phi_s METHOD SELECTOR
#   'voronoi'   -- standard Voronoi cell-volume fraction
#   'count'     -- N_s / N_total number fraction
#   'packing'   -- rho_s * v_bead packing fraction
# ==========================================================================
phi_s_method = 'voronoi'   # <-- change here

# ==========================================================================
#   p_p SOURCE SELECTOR
#
#   'group'   -- p_p = (1/3) tr(σ_s,partial) / φ_s   (group-based)
#   'ss_only' -- p_p = (1/3) tr(σ_s,ss) / φ_s         (pair/local ss-only)
#   'widom'   -- p_p = P_ext + (μ_total − μ_total,res) · ρ_s,res
#                Most physically robust.  P_ext auto-detected from the
#                total stress in the pure-solvent reservoir above the piston.
#                Requires mu_z_cavity_summary file (from cavity_widom.py).
#                Reference state: p_p,ref = P_ext (uniform — ε=0 equilibrium).
# ==========================================================================
p_p_source = 'widom'   # <-- change here:  'group', 'ss_only', or 'widom'

# ── Fallback guards ───────────────────────────────────────────────────────
if p_p_source == 'ss_only' and not has_ss_stresses:
    print(f'WARNING: p_p_source={p_p_source!r} but has_ss_stresses=False.  '
          'Falling back to group-based.')
    p_p_source = 'group'
if p_p_source == 'widom' and not has_mu_data:
    print(f'WARNING: p_p_source={p_p_source!r} but has_mu_data=False.  '
          'Falling back to group-based.\n'
          f'  Run:  python cavity_widom.py --traj <traj> --p-ext <P_ext> ...')
    p_p_source = 'group'

# ── phi_s selector ────────────────────────────────────────────────────────
_phi_opts = {
    'voronoi': (phi_s_final,     phi_s_ref,       'Voronoi'),
    'count':   (phi_s_cnt_fin_z, phi_s_cnt_ref_z, 'count-based  (N_s/N_total)'),
    'packing': (phi_s_pk_fin_z,  phi_s_pk_ref_z,
                f'packing fraction  (d_eff = {d_eff:.3f} sigma)'),
}
if phi_s_method not in _phi_opts:
    raise ValueError(f"phi_s_method must be one of {list(_phi_opts)}; got {phi_s_method!r}")
_phi_f, _phi_r, phi_s_label = _phi_opts[phi_s_method]
phi_s_final_safe = np.where(_phi_f > 1e-6, _phi_f, np.nan)
phi_s_ref_safe   = np.where(_phi_r > 1e-6, _phi_r, np.nan)
print(f'phi_s source: {phi_s_label}')
print(f'  phi_s_final (gel mean): {float(np.nanmean(_phi_f[phi_p_final > phi_gel_threshold])):.4f}')
print(f'  phi_s_ref   (gel mean): {float(np.nanmean(_phi_r[phi_p_ref   > phi_gel_threshold])):.4f}')
print()

# ── Poroelastic decomposition ─────────────────────────────────────────────
# LAMMPS sign convention: σ_tot = σ' + p_p I  (positive = compressive).
# σ'_zz = σ_p,zz + σ_s,zz − p_p
# Elastic increment Δσ' = σ'_final − σ'_ref  →  M = Δσ'_zz / ε_zz

if p_p_source == 'ss_only':
    pp_label = 'σ_s,ss  (pair/local, solvent–solvent only; no ½σ_ps, no kinetic)'
    p_p_final     = (1/3) * (sig_s_final_ss_zz + sig_s_final_ss_xx + sig_s_final_ss_yy) / phi_s_final_safe
    p_p_ref       = (1/3) * (sig_s_ref_ss_zz   + sig_s_ref_ss_xx   + sig_s_ref_ss_yy)   / phi_s_ref_safe
    p_p_final_err = np.full_like(z_coords, np.nan)   # uncertainty not tracked for stress sources
    p_p_ref_err   = np.full_like(z_coords, np.nan)

elif p_p_source == 'group':
    pp_label = 'σ_s,partial  (group-based: σ_ss + ½σ_ps + kinetic)'
    p_p_final     = (1/3) * (sig_s_final    + sig_s_final_xx + sig_s_final_yy) / phi_s_final_safe
    p_p_ref       = (1/3) * (sig_s_ref      + sig_s_ref_xx   + sig_s_ref_yy)   / phi_s_ref_safe
    p_p_final_err = np.full_like(z_coords, np.nan)
    p_p_ref_err   = np.full_like(z_coords, np.nan)

elif p_p_source == 'widom':
    # ── Widom pore pressure ───────────────────────────────────────────────
    # p_p(z) = P_ext + (μ_total(z) − μ_total,res) · ρ_s,res
    # P_ext is auto-detected from σ_tot in the pure-solvent reservoir so
    # that p_p,res = P_ext by construction and p_p → 0 outside the gel.
    _z_mu      = mu_summary['z_center']
    _mu_ex    = mu_summary['mu_ex_lastN']   # last-N-frame mean (N_AVG_FRAMES)
    _mu_ex_se = mu_summary['mu_se_lastN']   # last-N-frame stderr

    # Solvent number density on the Widom z-grid.
    # Prefer rho_s from the summary (same z-bins, same Widom trajectory,
    # piston excluded) to avoid the piston-face density spike that appears
    # in the stress-trajectory packing fraction.
    if 'rho_s_mean' in mu_summary and np.any(np.isfinite(mu_summary['rho_s_mean'])):
        _rho_s_mu = mu_summary['rho_s_mean']
    else:
        _rho_s_mu = interp1d(z_pk_fin, phi_s_pk_fin / v_bead,
                              bounds_error=False, fill_value=0.0)(_z_mu)
    with np.errstate(divide='ignore', invalid='ignore'):
        _mu_total_mu = _mu_ex + kT * np.log(np.where(_rho_s_mu > 0, _rho_s_mu, np.nan))

    # Reservoir: highest-density Widom bins (pure solvent above piston)
    _res_mu      = _rho_s_mu >= 0.85 * float(np.nanmax(_rho_s_mu))
    _mu_tot_res  = float(np.nanmean(_mu_total_mu[_res_mu]))
    _rho_s_res   = float(np.nanmean(_rho_s_mu[_res_mu]))

    # P_ext: from measured σ_tot in the same reservoir region.
    # In pure solvent above the piston, σ_p ≈ 0 and σ_s ≈ P_ext (barostat).
    _sigma_tot_full = sig_p_final + sig_s_final
    _sigma_tot_at_mu = interp1d(z_coords, _sigma_tot_full,
                                 bounds_error=False, fill_value=np.nan)(_z_mu)
    P_ext_detected = float(np.nanmean(_sigma_tot_at_mu[_res_mu]))

    # p_p on Widom z-grid; interpolate to stress z-grid
    _pp_mu     = P_ext_detected + (_mu_total_mu - _mu_tot_res) * _rho_s_res
    # Uncertainty: δp_p = ρ_s,res · δμ_ex  (linear propagation; ρ_s assumed exact)
    _pp_se_mu  = _rho_s_res * np.where(np.isfinite(_mu_ex_se), np.abs(_mu_ex_se), np.nan)

    _to_z = lambda arr: interp1d(_z_mu, arr, bounds_error=False, fill_value=np.nan)(z_coords)
    p_p_final     = _to_z(_pp_mu)
    p_p_final_err = _to_z(_pp_se_mu)

    # Reference: uniform P_ext (ε=0 equilibrium → p_p = P_ext everywhere)
    p_p_ref     = np.full_like(z_coords, P_ext_detected)
    p_p_ref_err = np.zeros_like(z_coords)

    pp_label = (f'cavity-Widom  [P_ext = {P_ext_detected:.4g} eps/sigma^3 (from reservoir sigma_tot),  '
                f'ρ_s,res = {_rho_s_res:.4g} sigma^-3]')
    print(f'  Widom pore pressure:')
    print(f'    P_ext (reservoir σ_tot)  = {P_ext_detected:.4f} eps/sigma^3')
    print(f'    μ_total,res              = {_mu_tot_res:.4f} ε')
    print(f'    ρ_s,res                  = {_rho_s_res:.4f} sigma^-3')
    _gel_m = phi_p_final > phi_gel_threshold
    print(f'    p_p gel mean (final)     = {float(np.nanmean(p_p_final[_gel_m])):.4f} eps/sigma^3')
    print(f'    p_p stderr (gel mean)  = {float(np.nanmean(p_p_final_err[_gel_m])):.4f} eps/sigma^3')

else:
    raise ValueError(f"p_p_source must be 'group', 'ss_only', or 'widom'; got {p_p_source!r}")

def _z(x):
    """Replace NaN with 0 so subtraction propagates cleanly outside the gel."""
    return np.where(np.isnan(x), 0.0, x)

# σ′ at each state: σ_tot − p_p I  (per-component)
sig_prime_final     = sig_p_final     + sig_s_final     - _z(p_p_final)
sig_prime_final_xx  = sig_p_final_xx  + sig_s_final_xx  - _z(p_p_final)
sig_prime_final_yy  = sig_p_final_yy  + sig_s_final_yy  - _z(p_p_final)

sig_prime_ref       = sig_p_ref       + sig_s_ref       - _z(p_p_ref)
sig_prime_ref_xx    = sig_p_ref_xx    + sig_s_ref_xx    - _z(p_p_ref)
sig_prime_ref_yy    = sig_p_ref_yy    + sig_s_ref_yy    - _z(p_p_ref)

# Elastic increment
sig_prime_elastic    = sig_prime_final    - sig_prime_ref
sig_prime_elastic_xx = sig_prime_final_xx - sig_prime_ref_xx
sig_prime_elastic_yy = sig_prime_final_yy - sig_prime_ref_yy

if abs(eps_final) < 1e-12:
    raise ValueError('Final strain ~= 0 -- cannot compute M')

# For Widom: sigma'_ref = 0 by construction (p_p,ref = P_ext = sigma_tot,ref),
# so use sigma'_final directly to avoid amplifying noise in the ref subtraction.
if p_p_source == 'widom':
    M_final = sig_prime_final / eps_final
else:
    M_final = sig_prime_elastic / eps_final

# ── Uncertainty propagation ───────────────────────────────────────────────
# δσ′_final ≈ δp_p  (stress measurement errors << μ_ex sampling error)
# δσ′_ref ≈ 0       (uniform P_ext reference, no μ_ex sampling)
# δ(Δσ′) = δσ′_final
# δM = δ(Δσ′) / |ε|
sig_prime_final_err  = np.where(np.isfinite(p_p_final_err), p_p_final_err, np.nan)
sig_prime_elastic_err = sig_prime_final_err
M_final_err          = np.where(np.isfinite(sig_prime_elastic_err),
                                 sig_prime_elastic_err / abs(eps_final), np.nan)

# ── Summary ───────────────────────────────────────────────────────────────
interior_mask = (phi_p_final > 0.05) & (phi_s_final > 0.05) & (phi_p_ref > 0.05) & (phi_s_ref > 0.05)

# M_mask: erode by 2 bins each side to exclude interface artifacts
_im_idx = np.where(interior_mask)[0]
if len(_im_idx) >= 5:
    M_mask = np.zeros(len(z_coords), dtype=bool)
    M_mask[_im_idx[2]:_im_idx[-2] + 1] = interior_mask[_im_idx[2]:_im_idx[-2] + 1]
else:
    M_mask = interior_mask.copy()
phi_p_mean    = float(np.nanmean(phi_p_final[interior_mask]))
phi_s_mean    = float(np.nanmean(phi_s_final[interior_mask]))

print(f"Pore-pressure source: {pp_label}")
print(f"φ_s averaged over last {n_phi_avg} traj frames; φ_s reference from frame {ref_frame_idx}")
print(f"Reference snapshot: {idx_ref}"
      + (f"  (t = {all_timesteps[idx_ref]})" if isinstance(idx_ref, int) else "  (ε=0 dedicated recording)"))
print(f"Final     snapshot: stress index {idx_final} (t = {all_timesteps[idx_final]})")
print()
print(f"  {'quantity':<28}  {'ref':>9}  {'final':>9}  {'Δ':>9}")
print(f"  {'-'*28}  {'-'*9}  {'-'*9}  {'-'*9}")
for label, q_r, q_f in [
    ("<σ_p_zz>",         sig_p_ref,           sig_p_final),
    ("<σ_s_zz>",         sig_s_ref,           sig_s_final),
    ("<σ_tot_zz>",       sig_p_ref+sig_s_ref, sig_p_final+sig_s_final),
    ("<p_p>",            p_p_ref,             p_p_final),
    ("<σ'_zz>",          sig_prime_ref,       sig_prime_final),
    ("<σ'_xx>",          sig_prime_ref_xx,    sig_prime_final_xx),
    ("<σ'_yy>",          sig_prime_ref_yy,    sig_prime_final_yy),
]:
    r = float(np.nanmean(q_r[interior_mask]))
    f = float(np.nanmean(q_f[interior_mask]))
    print(f"  {label:<28}  {r:>9.4f}  {f:>9.4f}  {f-r:>9.4f}")
print()
print(f"phi_p mean (interior, time-averaged): {phi_p_mean:.4f}")
print(f"phi_s mean (interior, time-averaged): {phi_s_mean:.4f}")
_M_label = "sigma'_zz" if p_p_source == "widom" else "Delta sigma'_zz"
print(f"M(z) = {_M_label} / eps_zz   (eps_zz = {eps_final:.4f})")
print(f"  <M>_gel (eroded interior) = {float(np.nanmean(M_final[M_mask])):.4f}")
_has_err = np.any(np.isfinite(M_final_err[M_mask]))
if _has_err:
    print(f"  <dM>_gel (from mu_ex stderr) = {float(np.nanmean(M_final_err[M_mask])):.4f}")


In [ ]:
# == Volume fraction comparison: count-based | Voronoi | packing fraction ==
# All three phi_s variants computed in cell 13.
# The active method (used for p_p and M above) is shown in the suptitle.

z_cnt_fin_norm = (z_cnt_fin - zlo_f) / Lz_f
z_pk_fin_norm  = (z_pk_fin  - zlo_f) / Lz_f

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
fig.suptitle(
    f'Volume Fraction Methods  --  Final Relaxed State:  {RUN_ID}\n'
    f'rho_total(gel) = {rho_total_gel:.3f} sigma^-3   '
    f'rho_s(gel) = {rho_s_gel:.3f} sigma^-3   '
    f'd_eff = {d_eff:.3f} sigma   '
    f'[active: {phi_s_method}]',
    fontsize=11, fontweight='bold')

panels = [
    (z_cnt_fin_norm, phi_p_cnt_fin, phi_s_cnt_fin,
     'Count-based',
     r'$\phi_s = N_s / N_\mathrm{total}$   (same-size beads: equals Voronoi)'),
    (z_vor_norm,     phi_p_vor,     phi_s_vor,
     'Voronoi',
     r'$\phi_s = V_s^\mathrm{Vor} / V_\mathrm{bin}$   (exact for polydisperse; = count for monodisperse)'),
    (z_pk_fin_norm,  phi_p_pk_fin,  phi_s_pk_fin,
     f'Packing fraction  ($d_{{\\rm eff}}$ = {d_eff:.3f}' + r'$\,\sigma$)',
     (r'$\phi_s = \rho_s \cdot (\pi/6)\,d_{{\rm eff}}^3$'
      rf',  $d_{{\rm eff}} = 2^{{1/6}}\sigma = {d_eff:.3f}$')),
]

for ax, (z_n, phi_p, phi_s, title, subtitle) in zip(axes, panels):
    ax.axvspan(z_gel_lo_norm, z_gel_hi_norm, color='lightgreen',
               alpha=0.15, zorder=0, label='gel')
    ax.plot(z_n, phi_p, 'o-', color='steelblue', ms=3, lw=1.5, label=r'$\phi_p$')
    ax.plot(z_n, phi_s, 's--', color='coral',     ms=3, lw=1.5, label=r'$\phi_s$')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=8)
    ax.text(0.5, -0.18, subtitle, transform=ax.transAxes,
            ha='center', fontsize=10, color='0.4', style='italic')
    ax.legend(fontsize=12)
    ax.set(xlabel=r'$z/L_z$', ylabel=r'$\phi(z)$', xlim=(0, 1))
    ax.grid(alpha=0.3)
    ax.axhline(0, color='k', lw=0.6, alpha=0.4)

out_vf = PLOT_DIR / f'volfrac_comparison_{sim_name}.png'
plt.savefig(out_vf, dpi=150, bbox_inches='tight')
print(f'Saved: {out_vf}')
plt.show()

# ── 2x2 final-state profiles (phi_s source shown in title) ─────────────────
fig2, axes2 = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
fig2.suptitle(
    f'Final Equilibrated State ({phi_s_method} phi): {RUN_ID}\n'
    f't = {all_timesteps[idx_final]}',
    fontsize=12, fontweight='bold')

ax = axes2[0, 0]
# Reference state (dashed)
ax.plot(z_norm, sig_p_ref,              '--', color='steelblue', lw=1.5, alpha=0.65,
        label=r'$\sigma_{p,zz}$ ref')
ax.plot(z_norm, sig_s_ref,              '--', color='coral',     lw=1.5, alpha=0.65,
        label=r'$\sigma_{s,zz}$ ref')
ax.plot(z_norm, sig_p_ref + sig_s_ref,  '--', color='black',     lw=1.5, alpha=0.65,
        label=r'$\sigma_{\rm tot}$ ref')
# Final state (solid)
ax.plot(z_norm, sig_p_final,                '-',  color='steelblue', lw=1.8,
        label=r'$\sigma_{p,zz}$')
ax.plot(z_norm, sig_s_final,                '-',  color='coral',     lw=1.8,
        label=r'$\sigma_{s,zz}$')
ax.plot(z_norm, sig_p_final + sig_s_final,  'o-', color='black',     lw=1.8, ms=3,
        label=r'$\sigma_{\rm tot}$')
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.legend(fontsize=10, ncol=2)
ax.set(xlabel='$z/L_z$', ylabel='stress (LJ)',
       title='(a) Partial stresses — ref (dashed) vs final', xlim=(0,1))
ax.grid(alpha=0.3)

ax = axes2[0, 1]
ax.plot(z_norm, _z(p_p_ref),   '--', color='steelblue', lw=2.0,
        label=r'reference ($\varepsilon=0$)')
ax.plot(z_norm, _z(p_p_final), 'o-', color='crimson',   ms=3, lw=2.0,
        label=r'final ($\varepsilon={:.3f}$)'.format(eps_final))
_pp_lo = _z(p_p_final) - np.where(np.isfinite(p_p_final_err), p_p_final_err, 0)
_pp_hi = _z(p_p_final) + np.where(np.isfinite(p_p_final_err), p_p_final_err, 0)
_pp_has_err = np.isfinite(p_p_final_err)
if _pp_has_err.any():
    ax.fill_between(z_norm, _pp_lo, _pp_hi, where=_pp_has_err,
                    color='crimson', alpha=0.18, lw=0, label=r'$\pm 1\sigma$ (μ_ex)')
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.legend(fontsize=11)
ax.set(xlabel='$z/L_z$', ylabel=r'$p_p$',
       title=f'(b) Pore pressure $p_p(z)$  [{phi_s_method}, {p_p_source}]', xlim=(0,1))
ax.grid(alpha=0.3)

ax = axes2[1, 0]
ax.plot(z_norm, sig_prime_final, 'o-', color='crimson', ms=3, lw=2.0,
        label=r"$\sigma'_{zz} = \sigma_{{\rm tot}} - p_p$")
_sp_has_err = np.isfinite(sig_prime_final_err)
if _sp_has_err.any():
    ax.fill_between(z_norm, sig_prime_final - sig_prime_final_err,
                    sig_prime_final + sig_prime_final_err,
                    where=_sp_has_err, color='crimson', alpha=0.18, lw=0,
                    label=r'$\pm 1\sigma$')
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.legend(fontsize=11)
ax.set(xlabel='$z/L_z$', ylabel=r"$\sigma'_{zz}$  (LJ)",
       title=r"(c) Network stress $\sigma'_{{zz}} = \sigma^t_{{zz}} - p_p$ (final)",
       xlim=(0,1))
ax.grid(alpha=0.3)

ax = axes2[1, 1]
ax.plot(z_norm, M_final, 'o-', color='crimson', ms=3)
_M_has_err = np.isfinite(M_final_err)
if _M_has_err.any():
    ax.fill_between(z_norm, M_final - M_final_err, M_final + M_final_err,
                    where=_M_has_err, color='crimson', alpha=0.18, lw=0,
                    label=r'$\pm 1\sigma$')
    ax.legend(fontsize=10)
ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax.set(xlabel='$z/L_z$', ylabel='$M$',
       title=r'(d) Longitudinal modulus $M = \sigma^{\prime}_{zz}/\varepsilon$', xlim=(0,1))
ax.grid(alpha=0.3)

out2 = PLOT_DIR / f'final_state_profiles_{sim_name}.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
print(f'Saved: {out2}')
plt.show()

# ── Network stress components ───────────────────────────────────────────────
sig_prime_xx_mean = float(np.nanmean(sig_prime_final_xx[interior_mask]))
sig_prime_yy_mean = float(np.nanmean(sig_prime_final_yy[interior_mask]))
sig_prime_zz_mean = float(np.nanmean(sig_prime_final[interior_mask]))
print(f"sigma'_xx mean (interior): {sig_prime_xx_mean:.4f}")
print(f"sigma'_yy mean (interior): {sig_prime_yy_mean:.4f}")
print(f"sigma'_zz mean (interior): {sig_prime_zz_mean:.4f}")

fig3, axes3 = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
fig3.suptitle(
    f'Final-State Network Stress Components (z-binned): {RUN_ID}\n'
    f't = {all_timesteps[idx_final]}',
    fontsize=12, fontweight='bold')

for ax, arr, clr, mean_val, ttl in zip(
    axes3,
    [sig_prime_final_xx, sig_prime_final_yy, sig_prime_final],
    ['steelblue', 'coral', 'purple'],
    [sig_prime_xx_mean,  sig_prime_yy_mean,  sig_prime_zz_mean],
    [r"(a) $\sigma'_{xx}(z)$", r"(b) $\sigma'_{yy}(z)$", r"(c) $\sigma'_{zz}(z)$"],
):
    ax.plot(z_norm, arr, 'o-', color=clr, ms=3, label=f'mean = {mean_val:.4g}')
    ax.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    ax.set(xlabel='$z/L_z$', ylabel=r"$\sigma'$", title=ttl, xlim=(0, 1))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=12)

out3 = PLOT_DIR / f'final_state_network_stress_components_{sim_name}.png'
plt.savefig(out3, dpi=150, bbox_inches='tight')
print(f'Saved: {out3}')
plt.show()


In [ ]:
# == Pore Pressure and Effective Network Stress: Reference vs Final ==========
#
# Panel (a): p_p(z) at the reference (epsilon=0) and final (epsilon~comp_percent)
#            states.  p_p = (1/3) * tr(sigma_s,ss) / phi_s
#            Uses ss-only stress when pairs_*.dump was loaded; falls back to
#            group-based sigma_s otherwise (see printout below figure).
#
# Panel (b): sigma'_zz(z) = sigma_{p,zz} + sigma_{s,zz} - p_p
#            Effective (network / drained-skeleton) stress.
#            Reference subtraction Delta_sigma' = sigma'_final - sigma'_ref
#            isolates the elastic increment used for the longitudinal modulus.
#
# Sign convention: LAMMPS virial convention (positive = compressive).

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
fig.suptitle(
    f'Pore Pressure and Network Stress  ($\\sigma_{{tot}} - p_p$)\n'
    f'{RUN_ID}   |   $\\varepsilon_{{zz}}^{{\\rm final}} = {eps_final:.3f}$',
    fontsize=16, fontweight='bold')

# Restrict to bins where the gel is present in either state
gel_mask = (phi_p_final > phi_gel_threshold) | (phi_p_ref > phi_gel_threshold)
z_plt = z_coords[gel_mask]

_gs = dict(color='lightgreen', alpha=0.15, zorder=0)

# ---- (a) Pore Pressure -------------------------------------------------------
ax = axes[0]
ax.axvspan(z_gel_lo, z_gel_hi, **_gs)
ax.plot(z_plt, _z(p_p_ref)[gel_mask],   '--', color='steelblue', lw=2.5,
        label=r'reference ($\varepsilon=0$)')
ax.plot(z_plt, _z(p_p_final)[gel_mask], '-',  color='crimson',   lw=2.5,
        label=r'final ($\varepsilon={:.3f}$)'.format(eps_final))
_pp17_err = np.where(np.isfinite(p_p_final_err), p_p_final_err, 0)[gel_mask]
_pp17_has = np.isfinite(p_p_final_err[gel_mask])
if _pp17_has.any():
    ax.fill_between(z_plt,
                    _z(p_p_final)[gel_mask] - _pp17_err,
                    _z(p_p_final)[gel_mask] + _pp17_err,
                    where=_pp17_has, color='crimson', alpha=0.18, lw=0)
ax.axhline(0, color='k', ls=':', lw=0.8, alpha=0.5)
ax.set_xlabel(r'$z\;(\sigma)$')
ax.set_ylabel(r'$p_p$  (LJ)')
ax.set_title(
    f'(a) Pore pressure  $p_p$  [{p_p_source}]',
    fontsize=15)
ax.legend(fontsize=14)
ax.grid(alpha=0.3)

# ---- (b) Network Stress ------------------------------------------------------
ax = axes[1]
ax.axvspan(z_gel_lo, z_gel_hi, **_gs)
ax.plot(z_plt, sig_prime_final[gel_mask], '-', color='crimson', lw=2.5,
        label=r"$\sigma'_{{zz}} = \sigma_{{\rm tot}} - p_p$  ($\varepsilon={:.3f}$)".format(eps_final))
_sp17_err = np.where(np.isfinite(sig_prime_final_err), sig_prime_final_err, 0)[gel_mask]
_sp17_has = np.isfinite(sig_prime_final_err[gel_mask])
if _sp17_has.any():
    ax.fill_between(z_plt,
                    sig_prime_final[gel_mask] - _sp17_err,
                    sig_prime_final[gel_mask] + _sp17_err,
                    where=_sp17_has, color='crimson', alpha=0.18, lw=0)
ax.axhline(0, color='k', ls=':', lw=0.8, alpha=0.5)
ax.set_xlabel(r'$z\;(\sigma)$')
ax.set_ylabel(r"$\sigma'_{zz}$  (LJ)")
ax.set_title(
    r"(b) Network stress  $\sigma'_{{zz}} = \sigma_{{p,zz}} + \sigma_{{s,zz}} - p_p$",
    fontsize=15)
ax.legend(fontsize=14)
ax.grid(alpha=0.3)

fig.text(0.5, -0.02, f'Pore-pressure source: {pp_label}',
         ha='center', fontsize=12, style='italic', color='gray')

out_pp = PLOT_DIR / f'pore_pressure_network_stress_{sim_name}.png'
plt.savefig(out_pp, dpi=150, bbox_inches='tight')
print(f'Saved: {out_pp}')
plt.show()

# Numerical summary
print(f'\nGel-interior means (phi_p > {phi_gel_threshold}):')
im = interior_mask
print(f"  p_p    reference  = {float(np.nanmean(_z(p_p_ref)[im])):.4f}")
print(f"  p_p    final      = {float(np.nanmean(_z(p_p_final)[im])):.4f}")
print(f"  sigma\'_zz  ref   = {float(np.nanmean(sig_prime_ref[im])):.4f}")
print(f"  sigma\'_zz  final = {float(np.nanmean(sig_prime_final[im])):.4f}")
print(f"  Delta sigma\'_zz  = {float(np.nanmean(sig_prime_elastic[im])):.4f}")
print(f"  Pore-pressure source: {pp_src}")
print(f"  M (eroded interior)  = {float(np.nanmean(M_final[M_mask])):.4f}")

In [ ]:
# == Total Solvent Chemical Potential μ_total = μ_ex + kT·ln(ρ_s) ===========
#
# Loaded from:
#   mu_z_cavity_summary_*  — time-averaged mean ± stderr  (MU_SUMMARY_FILE)
#   mu_z_cavity_*          — per-frame traces             (MU_FRAME_FILE)
#
# mu_total(z) = mu_ex(z) + kT*ln(rho_s(z)) should be spatially FLAT at
# equilibrium (chemical potential equalization between gel and reservoir).
#
# ρ_s source preference:
#   1. summary column 8 (rho_s_mean) — same Widom z-grid, piston excluded
#      → spike-free.  Requires cavity_widom.py >= 2026-05-27 re-run.
#   2. Packing fraction interpolated from stress trajectory (fallback).

if not has_mu_data:
    print('No chemical potential data available. Run cavity_widom.py to generate '
          f'{MU_SUMMARY_FILE.name}')
else:
    z_mu           = mu_summary['z_center']
    mu_ex_allframe = mu_summary['mu_ex_mean']   # kept for reference
    mu_se          = mu_summary['mu_ex_stderr']  # all-frame stderr
    N_AVG_FRAMES   = 3   # most-recent frames to use for the mean line

    # ── ρ_s on the Widom z-grid ──────────────────────────────────────────
    if 'rho_s_mean' in mu_summary and np.any(np.isfinite(mu_summary['rho_s_mean'])):
        rho_s_mu  = mu_summary['rho_s_mean']
        rho_s_src = 'summary col 8 (Widom traj, piston excluded)'
    else:
        rho_s_mu  = interp1d(z_pk_fin, phi_s_pk_fin / v_bead,
                             bounds_error=False, fill_value=np.nan)(z_mu)
        rho_s_src = 'packing fraction (fallback — re-run cavity_widom.py for col 8)'
    print(f'rho_s source: {rho_s_src}')

    with np.errstate(divide='ignore', invalid='ignore'):
        mu_ideal = kT * np.log(np.where(rho_s_mu > 0, rho_s_mu, np.nan))

    # Reservoir density mask (all-frame rho_s gives robust estimate)
    rho_max_mu  = float(np.nanmax(rho_s_mu))
    res_mask_mu = rho_s_mu >= 0.85 * rho_max_mu
    # mu_frames, mu_ex_lastN, mu_se_lastN, N_AVG_FRAMES all set in data-load cell


    # Use precomputed values from cell 7
    steps_sorted = sorted(mu_frames.keys())
    n_frames_mu  = len(steps_sorted)
    _n_plot      = min(N_AVG_FRAMES, n_frames_mu)
    mu_ex = mu_summary['mu_ex_lastN']
    mu_se = mu_summary['mu_se_lastN']

    mu_total = mu_ex + mu_ideal
    _res_fin = res_mask_mu & np.isfinite(mu_total)
    mu_total_res = float(np.nanmean(mu_total[_res_fin])) if _res_fin.any() else np.nan
    print(f'mu_total_res (last-{_n_plot}-frame mean) = {mu_total_res:.4f} eps')

    cmap_t = plt.cm.viridis

    def _fcolor(ki, n):
        return cmap_t(ki / max(n - 1, 1))

    # ── Figure: two panels ────────────────────────────────────────────────
    fig, axes_mu = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    fig.suptitle(
        f'Solvent Chemical Potential — cavity-Widom\n'
        f'{RUN_ID}   |   $\\varepsilon_{{{{zz}}}}^{{{{\\rm final}}}} = {eps_final:.3f}$',
        fontsize=15, fontweight='bold')

    # ── Panel (a): μ_ex(z) ───────────────────────────────────────────────
    _gs17 = dict(color='lightgreen', alpha=0.15, zorder=0)

    ax = axes_mu[0]
    ax.axvspan(z_gel_lo, z_gel_hi, **_gs17)
    for ki, step in enumerate(steps_sorted):
        zc_f, mu_f = mu_frames[step]
        is_last = (step == steps_sorted[-1])
        color   = 'darkorange' if is_last else _fcolor(ki, n_frames_mu)
        ax.plot(zc_f, mu_f, '-',
                color=color,
                lw=2.0 if is_last else 0.7,
                alpha=1.0 if is_last else 0.40,
                zorder=4 if is_last else 1)
    # Time-averaged mean ± stderr on top
    ax.plot(z_mu, mu_ex, 'o-', color='steelblue', lw=2.0, ms=4,
            label=r'$\langle\mu_{ex}\rangle$ (mean)', zorder=6)
    finite_se = np.isfinite(mu_se)
    if finite_se.any():
        ax.fill_between(z_mu[finite_se],
                        (mu_ex - mu_se)[finite_se],
                        (mu_ex + mu_se)[finite_se],
                        color='steelblue', alpha=0.25, lw=0, label=r'$\pm 1\sigma$')
    ax.axhline(0, color='k', ls=':', lw=0.8, alpha=0.5)
    if n_frames_mu > 0:
        ax.plot([], [], '-', color='darkorange', lw=2,
                label=f'last frame (step {steps_sorted[-1]:,})')
        if n_frames_mu > 1:
            ax.plot([], [], '-', color=_fcolor(0, n_frames_mu), lw=0.9, alpha=0.7,
                    label=f'earlier frames ({n_frames_mu - 1}  |  last {_n_plot} averaged)')
    ax.set_xlabel(r'$z\;(\sigma)$')
    ax.set_ylabel(r'$\mu_{ex}$  ($\varepsilon$)')
    ax.set_title(r'(a) Excess chemical potential  $\mu_{ex}(z)$', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)

    # ── Panel (b): μ_total(z) ────────────────────────────────────────────
    ax = axes_mu[1]
    ax.axvspan(z_gel_lo, z_gel_hi, **_gs17)
    for ki, step in enumerate(steps_sorted):
        zc_f, mu_f = mu_frames[step]
        is_last = (step == steps_sorted[-1])
        # per-frame μ_total: reuse mean ρ_s (same z-grid → direct broadcast)
        _rho_f = np.interp(zc_f, z_mu, rho_s_mu, left=np.nan, right=np.nan)
        with np.errstate(divide='ignore', invalid='ignore'):
            mu_tot_f = mu_f + kT * np.log(np.where(_rho_f > 0, _rho_f, np.nan))
        color = 'darkorange' if is_last else _fcolor(ki, n_frames_mu)
        ax.plot(zc_f, mu_tot_f, '-',
                color=color,
                lw=2.0 if is_last else 0.7,
                alpha=1.0 if is_last else 0.40,
                zorder=4 if is_last else 1)
    # Time-averaged mean on top
    ax.plot(z_mu, mu_total, 'o-', color='crimson', lw=2.0, ms=4,
            label=r'$\langle\mu_{\rm total}\rangle$ (mean)', zorder=6)
    ax.axhline(0, color='k', ls=':', lw=0.8, alpha=0.5)
    if np.isfinite(mu_total_res):
        ax.axhline(mu_total_res, color='gray', ls='--', lw=1.5, alpha=0.7,
                   label=fr'$\mu_{{\rm total,res}} = {mu_total_res:.3f}\,\varepsilon$')
        ax.fill_between(z_mu,
                        np.full_like(z_mu, mu_total_res - 0.02),
                        np.full_like(z_mu, mu_total_res + 0.02),
                        color='gray', alpha=0.08)
    if n_frames_mu > 0:
        ax.plot([], [], '-', color='darkorange', lw=2,
                label=f'last frame (step {steps_sorted[-1]:,})')
        if n_frames_mu > 1:
            ax.plot([], [], '-', color=_fcolor(0, n_frames_mu), lw=0.9, alpha=0.7,
                    label=f'earlier frames ({n_frames_mu - 1}  |  last {_n_plot} averaged)')
    ax.set_xlabel(r'$z\;(\sigma)$')
    ax.set_ylabel(r'$\mu_{\rm total}$  ($\varepsilon$)')
    ax.set_title(
        r'(b) Total chemical potential  $\mu_{\rm total}(z)$'
        '\n(flat at equilibrium)',
        fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(alpha=0.3)

    out_mu = PLOT_DIR / f'mu_total_{sim_name}.png'
    plt.savefig(out_mu, dpi=150, bbox_inches='tight')
    print(f'Saved: {out_mu}')
    plt.show()

    # ── Summary ───────────────────────────────────────────────────────────
    print(f'\nμ_ex  range: [{float(np.nanmin(mu_ex)):.4f}, {float(np.nanmax(mu_ex)):.4f}] eps')
    if np.isfinite(mu_total_res):
        _non_res = ~res_mask_mu & np.isfinite(mu_total)
        mu_total_gel_mu = float(np.nanmean(mu_total[_non_res])) if _non_res.any() else np.nan
        print(f'μ_total reservoir mean: {mu_total_res:.4f} eps')
        print(f'μ_total gel mean:       {mu_total_gel_mu:.4f} eps')
        print(f'Δμ_total (gel-res):     {mu_total_gel_mu - mu_total_res:.4f} eps  '
              f'(0.0 = full equilibrium)')


## Step 4: Gel Strain and Piston Force History

$\varepsilon_{zz}(t)$ from the Rg-based gel thickness and $F_{z,\text{piston}}(t)$ from the pairwise contact force on piston atoms. The force rises during compression, plateaus near `comp_percent`, then decays during relaxation — the decay timescale is directly related to $D_c$.

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 5.5), constrained_layout=True)
ax2 = ax1.twinx()

# ── Strain (left axis) ────────────────────────────────────────────────────
if ts_strain is not None:
    ax1.plot(ts_strain, eps_arr * 100, '-', color='steelblue', lw=2.5,
             label=r'$\varepsilon_{zz}$')
ax1.axhline(eps_final * 100, color='steelblue', ls=':', lw=1.5, alpha=0.55)
ax1.set_xlabel('Step')
ax1.set_ylabel(r'$\varepsilon_{zz}$ (%)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_ylim(bottom=0)

# ── Piston force (right axis) ─────────────────────────────────────────────
ax2.plot(steps_pf, F_piston, '-', color='firebrick', lw=2.0, alpha=0.85,
         label=r'$F_{z,\mathrm{piston}}$')
ax2.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)
ax2.set_ylabel(r'$F_{z,\mathrm{piston}}$ (LJ)', color='firebrick')
ax2.tick_params(axis='y', labelcolor='firebrick')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=16, loc='center right')

ax1.set_title(f'Gel Strain and Piston Force History  |  {sim_name}')
ax1.grid(alpha=0.3)

first_disp_ts = int(disp_ts[0])
idx_strain = np.argmin(np.abs(np.array(ts_strain, dtype=int) - first_disp_ts))
eps_at_start = float(eps_arr[idx_strain])
ax1.axvline(first_disp_ts, color='k', ls='--', lw=1.5, alpha=0.6,
            label=f'Displacement start (t={first_disp_ts})')

out_hist = PLOT_DIR / f'strain_force_history_{sim_name}.png'
plt.savefig(out_hist, dpi=150, bbox_inches='tight')
print(f'Saved: {out_hist}')
plt.show()


print(f'First displacement snapshot:  t = {first_disp_ts}')
print(f'Gel strain at that step:       ε = {eps_at_start:.4f}  ({eps_at_start*100:.2f}%)')
print(f'Target strain (comp_percent):  ε = {comp_percent}')
print(f'Final strain:  ε_zz = {eps_final:.4f}  ({eps_final*100:.2f}%)')
print(f'Gel thickness: L_0  = {L_final:.2f} sigma   ΔL = {dL_final:.2f} sigma')
print(f'Piston area = {piston_area:.2f} sigma^2  ')
print(f'Final piston force (step {steps_pf[-1]}): F_z = {F_piston[-1]:.4f} LJ')

# == Piston Force Plateau Detection ==========================================
# Sliding-window linear regression: the plateau starts at the first step from
# which the remainder of the time series has no significant linear trend
# (slope p-value > alpha).  Requires no distributional assumptions.
from scipy.stats import linregress as _linreg

_sa    = np.array(steps_pf, dtype=float)
_Fa    = np.array(F_piston,  dtype=float)
_n     = len(_sa)
_alpha = 0.05
_minw  = max(10, _n // 8)

plateau_idx  = _n - _minw
plateau_step = int(_sa[plateau_idx])
for _i in range(_n - _minw):
    _, _, _, _p, _ = _linreg(_sa[_i:], _Fa[_i:])
    if _p > _alpha:
        plateau_idx  = _i
        plateau_step = int(_sa[_i])
        break

_Fp   = _Fa[plateau_idx:]
_Fss  = float(_Fp.mean())
_Fstd = float(_Fp.std(ddof=1))
_Fse  = _Fstd / np.sqrt(len(_Fp))
_cv   = 100.0 * abs(_Fstd / _Fss) if abs(_Fss) > 1e-12 else np.nan

print('=' * 60)
print('PISTON FORCE PLATEAU ANALYSIS')
print('=' * 60)
print(f'Plateau detected at step:   {plateau_step:,}')
print(f'  ({plateau_idx} of {_n} points; '
      f'{100*len(_Fp)/_n:.0f}% of time series in plateau)')
print(f'Steady-state F_z mean:      {_Fss:.4f} LJ')
print(f'Steady-state F_z std:       {_Fstd:.4f} LJ')
print(f'Steady-state F_z stderr:    {_Fse:.4f} LJ  (n = {len(_Fp)})')
print(f'Coefficient of variation:   {_cv:.1f}%')

# Rolling statistics for the plot
_rw  = max(5, _n // 20)
_rm  = np.convolve(_Fa, np.ones(_rw) / _rw, mode='valid')
_rs  = np.array([_Fa[k:k+_rw].std(ddof=1) for k in range(len(_rm))])
_rst = _sa[_rw // 2: _rw // 2 + len(_rm)]

fig_plat, ax_plat = plt.subplots(figsize=(11, 5), constrained_layout=True)
ax_plat.plot(_sa, _Fa, '-', color='firebrick', lw=1.0, alpha=0.40,
             label=r'$F_{z,\rm piston}$ (raw)')
ax_plat.plot(_rst, _rm, '-', color='firebrick', lw=2.5,
             label=f'rolling mean (w={_rw})')
ax_plat.fill_between(_rst, _rm - _rs, _rm + _rs,
                     color='firebrick', alpha=0.20, lw=0,
                     label=r'$\pm 1\sigma$ rolling')
ax_plat.axvline(plateau_step, color='k', ls='--', lw=2.0,
                label=f'plateau start  (step {plateau_step:,})')
ax_plat.axhline(_Fss, color='darkgreen', ls='--', lw=1.5,
                label=f'<F_z> plateau = {_Fss:.3f} LJ')
ax_plat.fill_between(_sa, _Fss - _Fstd, _Fss + _Fstd,
                     color='darkgreen', alpha=0.12, lw=0)
ax_plat.set_xlabel('Step')
ax_plat.set_ylabel(r'$F_{z,\mathrm{piston}}$ (LJ)')
ax_plat.set_title(
    f'Piston Force Plateau Detection  |  {sim_name}\n'
    f'Plateau from step {plateau_step:,}:  '
    f'<F_z> = {_Fss:.3f} +/- {_Fstd:.3f} LJ  (CV = {_cv:.1f}%)',
    fontsize=13)
ax_plat.legend(fontsize=12)
ax_plat.grid(alpha=0.3)
out_plat = PLOT_DIR / f'force_plateau_{sim_name}.png'
plt.savefig(out_plat, dpi=150, bbox_inches='tight')
print(f'\nSaved: {out_plat}')
plt.show()

## Step 5: Longitudinal Modulus — Voronoi vs Piston Force

Two independent estimates of $M$:

- **Voronoi**: spatially resolved $M(z) = \sigma'_{zz}(z)/\varepsilon_{zz}$, gel-averaged with 95% CI from bin-to-bin spatial scatter.
- **Piston force**: $M_\text{piston} = P_\text{piston}/\varepsilon_{zz}$ where $P_\text{piston} = F_z / A$; single scalar from `fix print`.

Agreement validates the Voronoi decomposition. Deviations can indicate incomplete relaxation (piston still loading dynamically), non-affine deformation, or Voronoi tessellation errors.

In [ ]:
# ── Piston pressure P = F_z / A vs timestep ──────────────────────────────
P_piston_ts = F_piston / piston_area

fig, ax = plt.subplots(figsize=(11, 5.5), constrained_layout=True)

ax.plot(steps_pf, P_piston_ts, '-', color='firebrick', lw=2.0, alpha=0.85,
        label=r'$P = F_z / A$')
ax.axhline(0, color='k', ls='--', lw=0.8, alpha=0.4)

# Mark where compression ends and relaxation begins (peak pressure = piston freeze point)
peak_P_step = int(steps_pf[P_piston_ts.argmax()])
ax.axvline(peak_P_step, color='steelblue', ls='--', lw=1.5, alpha=0.7,
           label=f'Relaxation start / peak $P$ (t={peak_P_step})')

# Annotate final value
P_final_val = float(F_piston[-1]) / piston_area
ax.axhline(P_final_val, color='firebrick', ls=':', lw=1.5, alpha=0.55,
           label=f'Final $P = {P_final_val:.4f}$')

ax.set_xlabel('Step')
ax.set_ylabel(r'$P = F_z / A$ (LJ units / $\sigma^2$)')
ax.set_title(
    r'Piston Pressure $P = F_z / A$ vs Timestep'
    f'  |  {sim_name}\n'
    f'$A = {piston_area:.2f}\\,\\sigma^2$'
)
ax.legend(fontsize=16)
ax.grid(alpha=0.3)

out_P = PLOT_DIR / f'piston_pressure_history_{sim_name}.png'
plt.savefig(out_P, dpi=150, bbox_inches='tight')
print(f'Saved: {out_P}')
plt.show()

print(f'\nPiston pressure summary:')
print(f'  Peak P   = {P_piston_ts.max():.4f} LJ/sigma^2  at step {steps_pf[P_piston_ts.argmax()]}')
print(f'  Final P  = {P_final_val:.4f} LJ/sigma^2  (step {steps_pf[-1]})')
print(f'  eps_zz   = {eps_final:.4f}')


In [ ]:
print('=' * 70)
print('LONGITUDINAL MODULUS: VORONOI vs PISTON FORCE')
print('=' * 70)

# ── Method 1: Voronoi-based M (reference-subtracted, interior-trimmed) ──
# Gel mask: keep bins that are gel in BOTH the reference and final states,
# AND trim `edge_trim_bins` from each gel edge to avoid the support-side
# FENE tether spike and the piston-side density transition.
edge_trim_bins = 2

both_gel = (phi_p_final > phi_gel_threshold) & (phi_p_ref > phi_gel_threshold)
gel_idx_arr = np.where(both_gel)[0]
if len(gel_idx_arr) == 0:
    raise ValueError('No bins are gel in both states; loosen phi_gel_threshold.')

i_left  = gel_idx_arr[0]  + edge_trim_bins
i_right = gel_idx_arr[-1] - edge_trim_bins + 1
gel_mask = np.zeros_like(both_gel)
if i_right > i_left:
    gel_mask[i_left:i_right] = both_gel[i_left:i_right]

M_gel      = M_final[gel_mask]
M_gel      = M_gel[~np.isnan(M_gel)]
n_gel_bins = len(M_gel)

if n_gel_bins == 0:
    raise ValueError('No valid M values in trimmed gel region')

M_mean, M_ci_lo, M_ci_hi = mean_ci(M_gel, ci_level)
ci_pct = int(ci_level * 100)

print(f'\nMethod 1 -- Voronoi decomposition (Δσ′ / ε, edge_trim_bins={edge_trim_bins}):')
print(f'  Gel bins: {n_gel_bins}  (phi_p > {phi_gel_threshold} in both states)')
print(f'  M_gel range: [{M_gel.min():.4f}, {M_gel.max():.4f}]')
if n_gel_bins > 1:
    s = np.std(M_gel, ddof=1)
    sem = stats.sem(M_gel)
    df = n_gel_bins - 1
    t_crit = stats.t.ppf((1 + ci_level) / 2, df)
    print(f'  s = {s:.6f}   SEM = {sem:.6f}   t_crit(df={df}) = {t_crit:.4f}')
print(f'  M_voronoi = {M_mean:.4f}  [{M_ci_lo:.4f}, {M_ci_hi:.4f}]  ({ci_pct}% CI)')

# For reference, also report the OLD (no-reference-subtraction) value
M_old      = sig_prime_final / eps_final
M_old_gel  = M_old[gel_mask]
M_old_gel  = M_old_gel[~np.isnan(M_old_gel)]
if len(M_old_gel):
    print(f'  (Comparison) M_voronoi without ref subtraction = '
          f'{float(np.nanmean(M_old_gel)):.4f}')

# ── Method 2: Piston force ────────────────────────────────────────────────
# Average F_z over the same window as the final stress snapshot
# (width = nfreq_stress, estimated from spacing between last two snapshots).
ts_final  = all_timesteps[idx_final]
nfreq_est = int(all_timesteps[-1] - all_timesteps[-2]) if len(all_timesteps) >= 2 else int(ts_final)
win_lo    = ts_final - nfreq_est
win_hi    = ts_final
mask_win  = (steps_pf >= win_lo) & (steps_pf <= win_hi)
F_win     = F_piston[mask_win]

if len(F_win) >= 2:
    F_piston_final = float(np.mean(F_win))
    F_piston_std   = float(np.std(F_win, ddof=1))
else:
    idx_pf_final   = int(np.argmin(np.abs(steps_pf - ts_final)))
    F_piston_final = float(F_piston[idx_pf_final])
    F_piston_std   = 0.0
    print(f'  Warning: only {len(F_win)} force sample(s) in window; using nearest point')

P_piston_final = F_piston_final / piston_area
M_piston       = P_piston_final / eps_final
M_piston_err   = F_piston_std   / (piston_area * eps_final)

print(f'\nMethod 2 -- Piston force:')
print(f'  Window: steps {win_lo} – {win_hi}  ({np.sum(mask_win)} samples)')
print(f'  F_z = {F_piston_final:.4f} ± {F_piston_std:.4f} LJ  |  '
      f'A = {piston_area:.2f} sigma^2  |  '
      f'P = F/A = {P_piston_final:.4f}')
print(f'  M_piston = {M_piston:.4f} ± {M_piston_err:.4f}')

print(f'\nComparison:')
print(f'  M_voronoi (Δσ′/ε) = {M_mean:.4f}  [{M_ci_lo:.4f}, {M_ci_hi:.4f}]')
print(f'  M_piston          = {M_piston:.4f} ± {M_piston_err:.4f}')
print(f'  Ratio M_piston / M_voronoi = {M_piston / M_mean:.4f}')
print(f'\n  eps_zz = {eps_final:.4f}  |  L = {L_final:.2f}  |  dL = {dL_final:.2f}')
print('=' * 70)

# ── Comparison plot ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)

ax.errorbar([0], [M_mean],
            yerr=[[M_mean - M_ci_lo], [M_ci_hi - M_mean]],
            fmt='o', ms=12, color='steelblue', capsize=8, lw=2.5,
            label=f'Voronoi  $M = {M_mean:.3f}$\n{ci_pct}% CI: [{M_ci_lo:.3f}, {M_ci_hi:.3f}]')
ax.errorbar([1], [M_piston], yerr=[[M_piston_err], [M_piston_err]],
            fmt='s', ms=12, color='firebrick', capsize=8, lw=2.5,
            label=f'Piston  $M = {M_piston:.3f} \\pm {M_piston_err:.3f}$')

ax.axhline(M_mean,   color='steelblue', ls='--', lw=1.2, alpha=0.5)
ax.axhline(M_piston, color='firebrick', ls='--', lw=1.2, alpha=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Voronoi (Δσ′/ε)', 'Piston force'], fontsize=18)
ax.set_ylabel('$M$ (LJ units)')
ax.set_title(f'Longitudinal Modulus Comparison\n'
             f'{RUN_ID}  |  $\\varepsilon_{{zz}} = {eps_final:.3f}$')
ax.legend(fontsize=15, loc='upper right')
ax.set_xlim(-0.5, 1.5)
ax.grid(axis='y', alpha=0.3)

out_M = PLOT_DIR / f'M_comparison_{sim_name}.png'
plt.savefig(out_M, dpi=150, bbox_inches='tight')
print(f'Saved: {out_M}')
plt.show()


## Optional: Save Data

In [ ]:
out_data = PLOT_DIR / f'compression_modulus_final_{sim_name}.npz'
np.savez(out_data,
         # Coordinates
         z_coords=z_coords, z_norm=z_norm,
         z_vor=z_vor, z_vor_norm=z_vor_norm,
         # Timestep
         final_timestep=all_timesteps[idx_final],
         # Strain
         L=L_final, dL=dL_final, eps=eps_final,
         # Volume fractions (Voronoi)
         phi_p=phi_p_final, phi_s=phi_s_final,
         phi_p_vor=phi_p_vor, phi_s_vor=phi_s_vor,
         # Stress decomposition
         p_p=p_p_final, sigma_p_zz=sig_p_final, sigma_s_zz=sig_s_final,
         sigma_prime_zz=sig_prime_final,
         # Longitudinal modulus
         M=M_final,
         M_mean=M_mean, M_ci_lo=M_ci_lo, M_ci_hi=M_ci_hi,
         M_piston=M_piston, piston_area=piston_area,
         # Cooperative diffusivity
         Dc_fit=Dc_fit,
         # Metadata
         ci_level=ci_level, n_gel_bins=n_gel_bins,
         run_id=RUN_ID, sim_name=sim_name)
print(f'Saved: {out_data}')

## Step 6: Displacement Profile Evolution $u_z(z,t)$

Per-bin mean z-displacement from `fix ave/chunk`, colored early (dark) to late (bright). Edge bins with fewer than `Ncount_min` atoms are excluded. Reference is Phase 2 start so $u_z(z,0) = 0$; negative values indicate downward compression.

In [ ]:
Ncount_mean_all = np.mean(disp_Ncount, axis=0)
gel_mask_disp   = Ncount_mean_all > Ncount_min
z_gel_disp      = disp_z_raw[gel_mask_disp]

fig, ax = plt.subplots(figsize=(10, 7), constrained_layout=True)
norm_t  = Normalize(vmin=disp_ts[0], vmax=disp_ts[-1])
cmap    = plt.cm.viridis

for ts, snap in disp_snapshots:
    uz_i = snap[:, 3]
    ax.plot(z_gel_disp, uz_i[gel_mask_disp],
            '-', color=cmap(norm_t(ts)), lw=1.2, alpha=0.7)

ax.axhline(0, color='k', ls='--', lw=1, alpha=0.4)
ax.set(xlabel=r'$z$ ($\sigma$)', ylabel=r'$u_z$ ($\sigma$)',
       title=r'Polymer displacement $u_z(z,\,t)$  --  reference: Phase 2 start')
ax.grid(alpha=0.3)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm_t)
sm.set_array([])
fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02).set_label('Timestep')

out_uz = PLOT_DIR / f'uz_evolution_{sim_name}.png'
plt.savefig(out_uz, dpi=150, bbox_inches='tight')
print(f'Saved: {out_uz}')
plt.show()

## Step 7: Cooperative Diffusivity $D_c$ — Fourier Sine Fit

Fit $D_c$ to the even-mode sine series (Dirichlet BCs: zero displacement at both walls; zero IC):

$$u_z(z, t) = \sum_{k=1}^{N} A_k \left[1 - e^{-4\pi^2 k^2 \tau}\right] \sin\!\left(\frac{2\pi k\, z}{L_\text{gel}}\right), \qquad \tau = \frac{D_c\, t}{L_\text{gel}^2}$$

**Physical basis:**
- BCs: $u_z = 0$ at $\hat{z} = 0$ and $\hat{z} = 1$ — polymer cannot cross the walls (Dirichlet, not Neumann). No-flux on $\phi$ ($\partial\phi/\partial z = 0$ at walls) implies $\partial^2 u_z/\partial z^2 = 0$ at walls via $\phi - \bar\phi \approx -\bar\phi\,\partial u_z/\partial z$; together with the PDE this forces $\partial u_z/\partial t = 0$ at the walls, so the Dirichlet condition is self-consistently maintained.
- IC: $u_z = 0$ everywhere at $t = 0$ — displacement is measured from Phase 2 start. The $[1 - e^{-\lambda_k t}]$ factor satisfies this exactly without any virtual-time offset.
- Even modes $n = 2k$ (i.e.\ $\sin(2\pi k\hat{z})$) enforce antisymmetry about the gel midpoint ($u_z(\hat{z}=\tfrac{1}{2}) = 0$), consistent with the observed symmetric initial $\phi$ profile.

For fixed $D_c$, the amplitudes $\{A_k\}$ are solved exactly by linear least squares. $D_c$ is then found by scalar minimisation of the total squared residual over the first `frac_early` fraction of snapshots.

In [ ]:
# ── USER INPUTS ────────────────────────────────────────────────────────────
edge_margin  = 0       # extra gel-edge bins to drop before fitting
binWidth_Dc  = binWidth
frac_early   = 1     # use first frac_early fraction of snapshots
N_modes      = 5       # Fourier modes k = 1..N
Dc_bounds    = (1e-6, 1.0)
trim_bins    = 2       # interior trim for reference-value extraction
# ── END USER INPUTS ────────────────────────────────────────────────────────

# ── Match displacement snapshots to stress timesteps ──────────────────────
# Displacement and stress share nfreq so timesteps align by construction.
disp_ts_to_idx = {int(ts): i for i, ts in enumerate(disp_ts)}
uz_matched = []
Nc_matched = []
for ts in all_timesteps:
    if ts in disp_ts_to_idx:
        idx = disp_ts_to_idx[ts]
    else:
        nearest = int(min(disp_ts, key=lambda t: abs(t - ts)))
        print(f'  Warning: stress ts={ts} not in displacement, using nearest={nearest}')
        idx = disp_ts_to_idx[nearest]
    uz_matched.append(disp_uz[idx])
    Nc_matched.append(disp_Ncount[idx])
uz_matched = np.array(uz_matched)   # (n_stress, n_disp_bins)
Nc_matched = np.array(Nc_matched)

# ── Gel domain ────────────────────────────────────────────────────────────
gel_bins_d = np.where(Nc_matched[0] > Ncount_min)[0]
if len(gel_bins_d) == 0:
    raise RuntimeError(f'No gel bins with Ncount > {Ncount_min} in first snapshot')

i_left_d  = gel_bins_d[0]  + edge_margin
i_right_d = gel_bins_d[-1] - edge_margin + 1

z_gel_d     = disp_z_raw[i_left_d:i_right_d]
uz_gel      = uz_matched[:, i_left_d:i_right_d]   # (n_stress, n_gel_bins)
L_gel_sigma = (i_right_d - i_left_d) * binWidth_Dc
zeta_sigma  = z_gel_d - z_gel_d[0]
zhat        = zeta_sigma / L_gel_sigma

# ── Time arrays ───────────────────────────────────────────────────────────
times     = np.array(all_timesteps, dtype=float)
dt_lj_arr = (times - times[0]) * dt_lj   # elapsed time in LJ tau

# ── Early-snapshot selection ──────────────────────────────────────────────
max_dt_steps = (times[-1] - times[0]) * frac_early
early_idx    = np.where((dt_lj_arr <= max_dt_steps) & (dt_lj_arr > 0))[0]

if len(early_idx) == 0:
    raise RuntimeError(
        f'No early snapshots with frac_early={frac_early}. '
        f'Total steps={times[-1]-times[0]:.0f}. Try increasing frac_early.')

print(f'L_gel = {L_gel_sigma:.1f} sigma  |  {len(zhat)} bins  |  '
      f'{len(early_idx)} early snapshots  '
      f'(first {frac_early*100:.0f}% = {max_dt_steps*dt_lj:.1f} tau)')



In [ ]:
# ── Trim mask ─────────────────────────────────────────────────────────────
fit_mask        = np.ones(len(zhat), dtype=bool)
fit_mask[:trim_bins]  = False
fit_mask[-trim_bins:] = False
zhat_fit = zhat[fit_mask]
print(f'{np.sum(fit_mask)} bins in fit domain  (trim_bins={trim_bins} each side)')

# ── Fourier sine model ─────────────────────────────────────────────────────
# u_z(z,t) = sum_{k=1}^{N} A_k * [1 - exp(-4*pi^2*k^2 * tau)] * sin(2*pi*k*zh)
# where  tau = Dc * t / L_gel^2
# BCs: u_z = 0 at zh = 0 and zh = 1  (Dirichlet)
# IC:  u_z = 0 at t = 0              (reference: Phase 2 start)
# Even-n modes sin(2*pi*k*zh) are antisymmetric about zh = 0.5.

def build_basis(zh, Dc, t_lj_val):
    """Design matrix (n_pts, N_modes): column k is [1-exp(-4*pi^2*k^2*tau)]*sin(2*pi*k*zh)."""
    tau  = Dc * t_lj_val / L_gel_sigma**2
    cols = []
    for k in range(1, N_modes + 1):
        decay = 1.0 - np.exp(-4.0 * np.pi**2 * k**2 * tau)
        cols.append(decay * np.sin(2.0 * np.pi * k * zh))
    return np.column_stack(cols)   # (n_pts, N_modes)

def fit_amplitudes(Dc):
    """Linear least-squares solve for A_k given Dc (stacked over early snapshots)."""
    rows_y, rows_X = [], []
    for i in early_idx:
        rows_y.append(uz_gel[i][fit_mask])
        rows_X.append(build_basis(zhat_fit, Dc, dt_lj_arr[i]))
    A, _, _, _ = np.linalg.lstsq(
        np.vstack(rows_X), np.concatenate(rows_y), rcond=None)
    return A

def residual(Dc):
    A  = fit_amplitudes(Dc)
    ss = 0.0
    for i in early_idx:
        pred = build_basis(zhat_fit, Dc, dt_lj_arr[i]) @ A
        ss  += np.sum((pred - uz_gel[i][fit_mask])**2)
    return ss

def fourier_model(zh, Dc, A_k, t_lj_val):
    return build_basis(zh, Dc, t_lj_val) @ A_k

# True equilibrium: average last few frames
uz_equil = np.mean(uz_gel[-1:], axis=0)

# Get true A_k by sine decomposition of equilibrium profile
basis_inf = np.column_stack([np.sin(2*np.pi*k*zhat_fit) for k in range(1, N_modes+1)])
A_k_true, _, _, _ = np.linalg.lstsq(basis_inf, uz_equil[fit_mask], rcond=None)
print("True A_k from final snapshot:", A_k_true)
print(f"vs fit A_k: {A_k_fit}")

# Fit only Dc with A_k pinned to true equilibrium
def residual_pinned(Dc):
    ss = 0.0
    for i in early_idx:
        tau_i = Dc * dt_lj_arr[i] / L_gel_sigma**2
        pred = sum(A_k_true[k-1] * (1.0 - np.exp(-4*np.pi**2 * k**2 * tau_i))
                   * np.sin(2*np.pi*k*zhat_fit)
                   for k in range(1, N_modes+1))
        ss += np.sum((pred - uz_gel[i][fit_mask])**2)
    return ss



# ── Fit ───────────────────────────────────────────────────────────────────
# res      = minimize_scalar(residual, bounds=Dc_bounds, method='bounded')
# Dc_fit   = res.x
res = minimize_scalar(residual_pinned, bounds=Dc_bounds, method='bounded')
Dc_fit = res_pinned.x
A_k_fit  = fit_amplitudes(Dc_fit)

print(f'D_c = {Dc_fit:.4e} sigma^2/tau  (N_modes={N_modes})')
for k, A in enumerate(A_k_fit, 1):
    print(f'  A_{k} = {A:.4f} sigma  (amplitude of sin(2*pi*{k}*zh))')

    
tau_early_max = Dc_fit * dt_lj_arr[early_idx[-1]] / L_gel_sigma**2
tau_early_min = Dc_fit * dt_lj_arr[early_idx[0]] / L_gel_sigma**2
print(f"tau range in fit: {tau_early_min:.4f} – {tau_early_max:.4f}")
# Decay factors for k=1..5 at the last early snapshot:
for k in range(1, N_modes+1):
    d = 1 - np.exp(-4*np.pi**2 * k**2 * tau_early_max)
    print(f"  k={k}: [1-exp()] = {d:.3f}")
tau_final = Dc_fit * dt_lj_arr[-1] / L_gel_sigma**2
k1_sat_final = 1 - np.exp(-4 * np.pi**2 * tau_final)
print(f"tau at last snapshot: {tau_final:.4f}")
print(f"Mode-1 saturation at last snapshot: {k1_sat_final:.3f}")
print(f"Expected peak u_z at last snapshot: ~{A_k_fit[0] * k1_sat_final:.2f} sigma (mode-1 only)")


# ── R^2 ───────────────────────────────────────────────────────────────────
R2_per = []
for i in early_idx:
    y    = uz_gel[i][fit_mask]
    yhat = fourier_model(zhat_fit, Dc_fit, A_k_fit, dt_lj_arr[i])
    ss_r = np.sum((y - yhat)**2)
    ss_t = np.sum((y - np.mean(y))**2)
    R2_per.append(1.0 - ss_r / ss_t if ss_t > 1e-30 else np.nan)

uz_data_all = np.concatenate([uz_gel[i][fit_mask] for i in early_idx])
uz_pred_all = np.concatenate([fourier_model(zhat_fit, Dc_fit, A_k_fit, dt_lj_arr[i])
                               for i in early_idx])
ss_res = np.sum((uz_data_all - uz_pred_all)**2)
ss_tot = np.sum((uz_data_all - np.mean(uz_data_all))**2)
R2     = 1.0 - ss_res / ss_tot

print(f'R^2 = {R2:.6f}  (aggregate over early snapshots)')
print('R^2 per snapshot:')
for idx, r2 in zip(early_idx, R2_per):
    print(f'  step {times[idx]:.0f}  ->  R^2 = {r2:.3f}')

# ── Asymptotic profile (t -> inf) ─────────────────────────────────────────
zhat_fine = np.linspace(0, 1, 600)
uz_inf    = sum(A_k_fit[k - 1] * np.sin(2.0 * np.pi * k * zhat_fine)
                for k in range(1, N_modes + 1))
print(f'\nAsymptotic u_z extrema: min={uz_inf.min():.4f}, max={uz_inf.max():.4f} sigma')

# ── Sequential-difference plot  Δu_z(z, t_i) = u_z(t_i) − u_z(t_{i-1}) ──
fig_d, ax_d = plt.subplots(figsize=(10, 5), constrained_layout=True)
norm_t_d    = Normalize(vmin=times[early_idx[1]], vmax=times[early_idx[-1]])

for j in range(1, len(early_idx)):
    i_curr = early_idx[j]
    i_prev = early_idx[j - 1]
    delta  = uz_gel[i_curr][fit_mask] - uz_gel[i_prev][fit_mask]
    ax_d.plot(zhat_fit, delta, 'o-', color=cmap_fit(norm_t_d(times[i_curr])),
              ms=3, lw=1.2, alpha=0.7)

ax_d.axhline(0, color='k', ls='--', lw=1, alpha=0.5)
ax_d.set(xlabel=r'$\hat{z} = z / L_\mathrm{gel}$',
         ylabel=r'$u_z(t_i) - u_z(t_{i-1})$  ($\sigma$)',
         xlim=(0, 1))
ax_d.grid(alpha=0.3)
ax_d.set_title(r'Sequential displacement increments $\Delta u_z = u_z(t_i) - u_z(t_{i-1})$')

sm_d = plt.cm.ScalarMappable(cmap=cmap_fit, norm=norm_t_d)
sm_d.set_array([])
fig_d.colorbar(sm_d, ax=ax_d, fraction=0.025, pad=0.02).set_label('Timestep $t_i$')

out_delta = PLOT_DIR / f'Dc_uz_increments_{sim_name}.png'
plt.savefig(out_delta, dpi=150, bbox_inches='tight')
print(f'Saved: {out_delta}')
plt.show()

# ── Plot ──────────────────────────────────────────────────────────────────
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)
norm_t   = Normalize(vmin=times[early_idx[0]], vmax=times[early_idx[-1]])
cmap_fit = plt.cm.viridis

for i in early_idx:
    c = cmap_fit(norm_t(times[i]))
    ax_l.plot(zhat, uz_gel[i], 'o-', color=c, ms=3, alpha=0.6)
    ax_r.plot(zhat, uz_gel[i], 'o-', color=c, ms=3, alpha=0.35)
    ax_r.plot(zhat_fine, fourier_model(zhat_fine, Dc_fit, A_k_fit, dt_lj_arr[i]),
              '-', color=c, lw=2.0)

ax_r.plot(zhat_fine, uz_inf, 'k--', lw=1.8, label=r'$u_z(t\to\infty)$')

for ax in (ax_l, ax_r):
    ax.axhline(0, color='steelblue', ls=':', lw=1.5, label=r'$u_z = 0$')
    ax.set(xlabel=r'$\hat{z} = z / L_\mathrm{gel}$',
           ylabel=r'$u_z$ ($\sigma$)', xlim=(0, 1))
    ax.grid(alpha=0.3)

ax_l.legend(fontsize=14)
ax_l.set_title(r'Raw $u_z(z,\,t)$ — early snapshots')
ax_r.legend(fontsize=12)
ax_r.set_title(rf'Sine fit ($N={N_modes}$):  $D_c = {Dc_fit:.2e}\ \sigma^2/\tau$,  $R^2 = {R2:.3f}$')

sm = plt.cm.ScalarMappable(cmap=cmap_fit, norm=norm_t)
sm.set_array([])
fig.colorbar(sm, ax=[ax_l, ax_r], fraction=0.015, pad=0.04).set_label('Timestep')
fig.suptitle(f'Cooperative diffusivity fit  |  {sim_name}', fontsize=12, fontweight='bold')

out_fourier = PLOT_DIR / f'Dc_fourier_fit_{sim_name}.png'
plt.savefig(out_fourier, dpi=150, bbox_inches='tight')
print(f'\nSaved: {out_fourier}')
plt.show()
